# Evaluation — every number the evaluation chapter cites

Each block below states **the question it answers**, **how it is measured**, the **result**, and
a **takeaway that is falsifiable by the numbers in its own result dict**. Nothing here is a
literal copied from another notebook: a figure that cannot be recomputed in this file is not
reported in this file.

The research questions are **faithfulness, calibration and abstention** — formal system
properties with established metrics (execution accuracy for text-to-SQL, ECE for calibration,
P@k / MRR / nDCG for retrieval, citation precision for grounded generation). Gold labels are
**researcher-authored once** and frozen.

## Two things every metric must declare

**1. `corpus_dependent`.** The corpus was re-imported in September 2026 and is much larger
(155,164 indexed passages; 1,574 ECHR cases). A metric computed from a frozen *question* set is
unaffected. A metric computed from frozen *judgments of retrieved passages* is not, because the
ranking changed underneath the labels. So every metric carries a flag:

- `corpus_dependent: false` — computed from a frozen question set, or from per-case labels that
  no change of ranking can touch. The re-import cannot invalidate it.
- `corpus_dependent: true` — computed against the corpus as it currently stands. Valid when
  recomputed here; **stale** when it rests on judgments made against a superseded index, in
  which case `status` says `STALE` and the number is labelled with the index it was judged on.

**2. `backend` and `model`, for anything an LLM produced.** A hosted provider can change the
weights behind a model string without notice, so a hosted number is not reproducible the way a
local model at temperature 0 is. The determinism metric therefore covers the **deterministic**
paths only; the LLM paths are reproducible *by cache*, not by construction.

## What is deliberately not here

The hand-built rules column `alienation_alleged` is being retired and is **not** re-validated
against the new corpus. It survives in two places, both labelled: the calibration metric, because it is the
one field with reviewed labels today and that metric is about the *calibration protocol* rather
than the field; and the evidence-grounding metric, whose anti-fabrication check runs over the only evidence column
the extraction table currently carries. Both name the column in their result dict. The
content-field path that matters going forward is `field_factory.ipynb` → `field_deploy.ipynb`
→ the deployed parquet + sidecar meta. The content-field metrics are written against **any**
field the factory ships, score every field that carries adjudicated labels, and headline the
**deployed** one; a field that cannot be scored is listed with the reason, and if none can be,
all three register as `pending`.


In [1]:
import contextlib
import io
import json
import math
import os
import re
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

if "." not in sys.path:
    sys.path.insert(0, ".")                 # notebooks run from src/
# ONE definition of each shared statistic. `ece` takes n_bins as a REQUIRED argument: two local
# copies with different defaults (10 here, 5 in extraction_validation.ipynb) is how a 10-bin raw
# figure came to be printed next to a 5-bin calibrated one as if it were a before/after.
from eval_metrics import bin_stats, count_bias, ece, oof_isotonic, prf1, wilson

# Boot the DEPLOYED pipeline exactly as ask_web.py does, so evaluation cannot drift from the
# shipped system. The dispatcher cell is matched on `def ask_anything` and no longer on
# `def _h_alienation`: that branch is being retired, and a boot marker naming it would stop
# finding the cell the day it goes.
_nb = json.loads(Path("ask.ipynb").read_text())
_BOOT = ["RAG_NB   = Path", "_swiss = _json.loads", "def _h_diachronic", "def ask_anything"]
with contextlib.redirect_stdout(io.StringIO()):
    for _c in _nb["cells"]:
        if _c["cell_type"] == "code" and any(m in "".join(_c["source"]) for m in _BOOT):
            exec("".join(_c["source"]), globals())
assert "aggregate_trigger" in globals() and "nl2sql" in globals() and con is not None
assert "ask_anything" in globals(), "dispatcher cell not booted -- check the _BOOT markers"

DATA, REPORTS, FIGURES = DATA_DIR, Path("../reports"), Path("../figures")
FIGURES.mkdir(exist_ok=True)
RUN_AT = datetime.now().isoformat(timespec="seconds")
N_INDEX = int(index.ntotal)
N_ECHR = int(con.execute("SELECT COUNT(*) FROM echr_meta").fetchone()[0])
print(f"pipeline booted | {N_INDEX} passages indexed | {N_ECHR} ECHR cases in the query DB")

# --- which backend served which call site --------------------------------------------------
# The four LLM call sites do NOT share a backend, and should not:
#   Bucket-1 generation   hosted   ~7k-token prompt; ~9-15 min per answer on this CPU
#   nl2sql()              local    constant KV-cached prefix (~15 s warm) AND the subject of a
#                                  thesis finding -- see the synthetic-QA metric
#   field_factory/deploy  local    per-case prompts, no prefix reuse: the open bottleneck
#   _dia_llm() (Bucket 4) local    small, not evaluated here
_probe = io.StringIO()
with contextlib.redirect_stdout(_probe):
    GEN_READY = bool(gen_backend_check(probe=True))
BACKEND_REPORT = _probe.getvalue().strip()
GEN_MODEL_STR = GEN_MODEL if GEN_BACKEND == "ollama" else GEN_API_MODEL
print("\n--- generation backend: gen_backend_check(probe=True) ---")
print(BACKEND_REPORT)
print(f"--- query backend: nl2sql -> ollama/{NL2SQL_MODEL} (reachable at boot: {OLLAMA_OK}) ---")

RUN_ENV = {
    "run_at": RUN_AT,
    "index_passages": N_INDEX,
    "echr_cases_in_db": N_ECHR,
    "generation": {"backend": GEN_BACKEND, "model": GEN_MODEL_STR, "ready": GEN_READY},
    "nl2sql": {"backend": "ollama", "model": NL2SQL_MODEL, "reachable_at_boot": bool(OLLAMA_OK)},
    "gen_backend_check": BACKEND_REPORT,
    "deployed_content_fields": sorted(DEPLOYED_FIELDS),
}

# --- the registry --------------------------------------------------------------------------
RESULTS, ORDER = {}, []


def reg(key, name, question, how, value, example, tier, takeaway, *,
        corpus_dependent, headline, status="computed", backend=None, model=None):
    """Register one metric.

    corpus_dependent : True  = computed against the corpus as it currently stands;
                       False = frozen question set / per-case labels, immune to the re-import.
    status           : "computed" | "STALE" (number stands, but was judged on a superseded
                       index) | "pending" (could not run; the reason is in the result dict).
    headline         : (label, value) shown in the summary table. It must be a key that is
                       actually present in `value` -- no metric is summarised by a number that
                       is not in its own result dict.
    backend / model  : required for anything an LLM produced; None for deterministic metrics.

    `name` carries NO number. The registry numbers metrics by registration order, so inserting
    one never requires renumbering the others -- and prose refers to metrics by name rather
    than by number for the same reason.
    """
    assert headline[0] in value, f"{key}: headline {headline[0]!r} is not in the result dict"
    assert not re.match(r"^\d+\.", name), f"{key}: reg() numbers metrics; drop the prefix"
    if key not in ORDER:
        ORDER.append(key)
    name = f"{ORDER.index(key) + 1}. {name}"
    RESULTS[key] = {"name": name, "what": question, "how": how, "value": value,
                    "example": example, "tier": tier, "takeaway": takeaway,
                    "corpus_dependent": bool(corpus_dependent), "status": status,
                    "headline_key": headline[0], "headline": headline[1],
                    "backend": backend, "model": model, "computed_at": RUN_AT}
    print("=" * 80)
    print(name.upper() + (f"   [{status}]" if status != "computed" else ""))
    print("  QUESTION IT ANSWERS :", question)
    print("  HOW MEASURED        :", how)
    print(f"  CORPUS-DEPENDENT    : {bool(corpus_dependent)}"
          + (f" | BACKEND: {backend} / {model}" if backend else ""))
    print("  RESULT:")
    for k, v in value.items():
        print(f"      - {k}: {v}")
    if example:
        print("  EXAMPLE :", str(example))
    print("  TAKEAWAY:", takeaway)


def reg_pending(key, name, question, how, reason, tier, *, corpus_dependent, unblocks=""):
    """A metric that could not run is a VISIBLE hole, not an absent row."""
    reg(key, name, question, how,
        {"pending": True, "pending_reason": reason, "unblocked_by": unblocks},
        None, tier,
        f"NOT MEASURED. {reason} Nothing may be claimed about this until it runs.",
        corpus_dependent=corpus_dependent, headline=("pending", "pending"), status="pending")


def pct(x, nd=3):
    return None if x is None or (isinstance(x, float) and math.isnan(x)) else round(float(x), nd)


def ci(k, n):
    """95% Wilson interval as a rounded [lo, hi] pair, for a rate k/n."""
    lo, hi = wilson(int(k), int(n))
    return [pct(lo), pct(hi)]


pipeline booted | 155164 passages indexed | 1574 ECHR cases in the query DB

--- generation backend: gen_backend_check(probe=True) ---
backend=remote | model=openai/gpt-oss-120b @ https://api.groq.com/openai/v1
  key=present (env GROQ_API_KEY) | budget=17920 chars | ready=True (ok)
  probe: 0.6s -> 'OK' (finish=stop)
--- query backend: nl2sql -> ollama/qwen2.5-coder:3b (reachable at boot: True) ---


## Tier 1 — Gold-label metrics
Researcher-authored labels, created once and frozen, scored with standard metrics.


In [1]:
# ROUTING ---------------------------------------------------------------------------------
gq = pd.read_csv(DATA / "router_gold_questions.csv")


def predict_bucket(q):
    """The DEPLOYED dispatch decision, not a re-implementation of it.

    `ask_anything()` asks four questions in this order and so does this: diachronic trigger ->
    aggregate trigger -> deployed-field match -> deny-list. Every branch calls the shipped
    function. The previous version of this cell carried its own `re.search(r"alienat|entfremd")`
    mirroring the `_h_alienation` branch; that branch is being retired, so the mirror would have
    gone on scoring a capability the system no longer has. No field regex is hardcoded here:
    `_match_deployed_field` reads the sidecar meta of whatever `field_deploy.ipynb` has shipped.

    Returns (bucket, the trigger that did or did not fire).
    """
    if _diachronic_trigger(q):
        return 4, "diachronic trigger (change-over-time + time expression)"
    trig = aggregate_trigger(q)
    if not trig:
        return 1, "no aggregate trigger -> retrieval"
    fld = _match_deployed_field(q)
    if fld:
        return 3, f"aggregate {trig!r} -> deployed field `{fld}`"
    if _UNEXTRACTED.search(q):
        return 3, f"aggregate {trig!r} -> deny-list (known-unextracted concept)"
    return 2, f"aggregate {trig!r} -> NL->SQL"


_pred = [predict_bucket(q) for q in gq.question]
gq["pred"] = [p for p, _ in _pred]
gq["why"] = [w for _, w in _pred]
# The route is three-way, not binary: retrieval / aggregate (query layer) / diachronic.
# Buckets 2 and 3 share the "aggregate" route because the dispatcher does not separate them --
# the query layer does, downstream, which is what `bucket_ok` below scores.
_ROUTE_OF = {1: "retrieval", 2: "aggregate", 3: "aggregate", 4: "diachronic"}
gq["pred_route"] = gq.pred.map(_ROUTE_OF)
gq["route_ok"] = gq.pred_route == gq.gold_route
gq["bucket_ok"] = gq.pred == gq.bucket

n = len(gq)
route_k = int(gq.route_ok.sum())
bucket_k = int(gq.bucket_ok.sum())

# the two error DIRECTIONS -- accuracy alone averages a fabricated statistic with a mild
# inconvenience, and they are not the same kind of mistake
b3, b1, b4 = gq[gq.bucket == 3], gq[gq.bucket == 1], gq[gq.bucket == 4]
guard = gq[gq.bucket != 1]                            # all that top-k generation must not answer
fab_k = int((b3.pred == 1).sum())                     # content aggregate sent to retrieval
nar_k = int((b4.pred == 1).sum())                     # change-over-time sent to retrieval
abst_k = int((b1.pred != 1).sum())                    # explanatory question refused
guard_k = int((guard.pred != 1).sum())                # kept away from top-k generation
b3_to_sql = int((b3.pred == 2).sum())                 # content aggregate handed to NL->SQL

print(f"routing accuracy (3 routes: retrieval/aggregate/diachronic): {route_k}/{n} = "
      f"{route_k/n:.3f} CI {ci(route_k, n)}")
print(f"strict 4-way bucket accuracy              : {bucket_k}/{n} = {bucket_k/n:.3f} "
      f"CI {ci(bucket_k, n)}")
print(f"FABRICATION RISK (gold 3 -> retrieval)    : {fab_k}/{len(b3)} = {fab_k/len(b3):.3f} "
      f"CI {ci(fab_k, len(b3))}")
print(f"NARRATIVE RISK   (gold 4 -> retrieval)    : {nar_k}/{len(b4)} = {nar_k/len(b4):.3f} "
      f"CI {ci(nar_k, len(b4))}")
print(f"OVER-ABSTENTION  (gold 1 -> aggregate)    : {abst_k}/{len(b1)} = {abst_k/len(b1):.3f} "
      f"CI {ci(abst_k, len(b1))}")
print(f"GENERATION GUARD (gold 2/3/4 kept off it) : {guard_k}/{len(guard)} = "
      f"{guard_k/len(guard):.3f} CI {ci(guard_k, len(guard))}")

bxl = gq.groupby(["bucket", "lang"]).route_ok.agg(["mean", "size"]).round(3)
print("\nrouting accuracy per gold bucket x language:")
print(bxl.rename(columns={"mean": "acc", "size": "n"}).to_string())

mis = gq[~gq.route_ok]
print(f"\nmisrouted questions, verbatim ({len(mis)}):")
for r in mis.itertuples():
    print(f"  [gold b{r.bucket} {r.lang}] routed {r.pred_route} (bucket {r.pred})")
    print(f"    Q: {r.question}")
    print(f"    trigger: {r.why}")
# The dispatcher's deny-list is not the only one: nl2sql() checks its own, wider UNEXTRACTED_RE
# before it calls any model, so a question that reaches Bucket 2 can still be refused without a
# single token being generated. That second net is deterministic, so it is measured here for
# free -- what the MODEL then does with the questions that get past it is not measured by this
# metric and must not be asserted by it.
# Bucket 4 is the newest branch and the only one whose trigger is monolingual:
# `aggregate_trigger` carries German patterns, `_DIA_CHANGE`/`_DIA_TIME` do not. That is a
# lexicon gap in one deployed regex, so it is reported per language rather than averaged away.
b4_de_miss = int(((b4.lang == "de") & (b4.pred == 1)).sum())
if len(b4):
    print(f"\nBucket 4 by language: "
          + ", ".join(f"{l} {int(g.bucket_ok.sum())}/{len(g)}"
                      for l, g in b4.groupby("lang")))
    if b4_de_miss:
        print(f"  {b4_de_miss} German change-over-time question(s) miss the diachronic branch "
              f"entirely -- `_DIA_CHANGE`/`_DIA_TIME` are English-only, unlike the bilingual "
              f"aggregate patterns. Reported, not patched: the fix moves the headline.")

b3_sql = gq[(gq.bucket == 3) & (gq.pred == 2)]
b3_denied = int(b3_sql.question.map(lambda q: bool(UNEXTRACTED_RE.search(q))).sum())
if b3_to_sql:
    _depf = sorted(DEPLOYED_FIELDS)
    _why = (f"the retired alienation branch is not replaced for them by the deployed field(s) "
            f"{_depf}, whose question terms none of them match"
            if _depf else "the alienation branch is retired and no field is deployed to replace it")
    print(f"\nnote: {b3_to_sql} gold-bucket-3 question(s) now reach the NL->SQL path -- "
          f"{_why}. Of those, {b3_denied} are refused outright by nl2sql()'s own "
          f"deny-list before any model call; the remaining {b3_to_sql - b3_denied} reach the "
          f"translator, and what its EXPLAIN/filter/GROUP BY nets do with them is NOT measured "
          f"here (the synthetic-QA and trap-set metrics probe those nets on other questions).")

reg("routing", "Routing accuracy and the error directions",
    "Does the system send each question to the capability that can answer it faithfully, and "
    "when it errs, does it err towards refusing or towards fabricating?",
    f"The DEPLOYED decision functions (_diachronic_trigger, aggregate_trigger, "
    f"_match_deployed_field, _UNEXTRACTED) called in ask_anything()'s own order over {n} frozen "
    f"researcher-labelled gold questions ({gq.groupby('bucket').size().min()} per bucket across "
    f"{gq.bucket.nunique()} buckets, {int((gq.lang == 'en').sum())} EN / "
    f"{int((gq.lang == 'de').sum())} DE). No LLM, no regex written for the evaluation. 95% "
    f"intervals are Wilson, not normal -- the normal approximation has zero width at p=0 and "
    f"would report a fabrication risk of 0.00 +/- 0.00.",
    {"n_questions": n,
     "buckets": sorted(int(b) for b in gq.bucket.unique()),
     "route_accuracy": pct(route_k / n),
     "route_accuracy_ci95": ci(route_k, n),
     "route_errors": n - route_k,
     "strict_4-way_accuracy": pct(bucket_k / n),
     "strict_4-way_accuracy_ci95": ci(bucket_k, n),
     "generation_guard": pct(guard_k / len(guard)),
     "generation_guard_n": f"{guard_k}/{len(guard)}",
     "generation_guard_ci95": ci(guard_k, len(guard)),
     "fabrication_risk": pct(fab_k / len(b3)), "fabrication_risk_n": f"{fab_k}/{len(b3)}",
     "fabrication_risk_ci95": ci(fab_k, len(b3)),
     "narrative_fabrication_risk": pct(nar_k / len(b4)),
     "narrative_fabrication_risk_n": f"{nar_k}/{len(b4)}",
     "narrative_fabrication_risk_ci95": ci(nar_k, len(b4)),
     "bucket4_german_missed": b4_de_miss,
     "over_abstention": pct(abst_k / len(b1)), "over_abstention_n": f"{abst_k}/{len(b1)}",
     "over_abstention_ci95": ci(abst_k, len(b1)),
     "gold3_routed_to_nl2sql": b3_to_sql,
     "gold3_at_nl2sql_refused_by_its_own_denylist": b3_denied,
     "gold3_at_nl2sql_reaching_the_model": b3_to_sql - b3_denied,
     "accuracy_by_bucket_x_language": {f"b{b}_{l}": pct(v) for (b, l), v in
                                       gq.groupby(["bucket", "lang"]).route_ok.mean().items()},
     "misrouted": [{"question": r.question, "gold_bucket": int(r.bucket), "lang": r.lang,
                    "predicted_bucket": int(r.pred), "trigger": r.why}
                   for r in mis.itertuples()]},
    "'positive obligations to enforce contact rights' -> retrieval; 'how many cases against "
    "Poland' -> query layer; 'how has the wording changed between 2000 and 2020' -> diachronic",
    "gold",
    f"The load-bearing decision -- keeping questions top-k cannot answer away from generation "
    f"-- holds {guard_k}/{len(guard)} times (CI {ci(guard_k, len(guard))}); the three-way route "
    f"is right {route_k}/{n} (CI {ci(route_k, n)}). The error directions are not symmetric and "
    f"are reported separately: {fab_k}/{len(b3)} content-aggregate and {nar_k}/{len(b4)} "
    f"change-over-time questions reached retrieval, {abst_k}/{len(b1)} explanatory questions "
    f"were refused. The change-over-time miss is concentrated in German "
    f"({b4_de_miss} of it), where the diachronic trigger has no patterns at all -- a named "
    f"lexicon gap, not a property of the design. Per-bucket n={len(b1)} is the binding "
    f"constraint: the intervals, not the point estimates, are what those rates support.",
    corpus_dependent=False,
    headline=("route_accuracy", pct(route_k / n)))


routing accuracy (3 routes: retrieval/aggregate/diachronic): 65/80 = 0.812 CI [0.713, 0.883]
strict 4-way bucket accuracy              : 54/80 = 0.675 CI [0.566, 0.768]
FABRICATION RISK (gold 3 -> retrieval)    : 2/20 = 0.100 CI [0.028, 0.301]
NARRATIVE RISK   (gold 4 -> retrieval)    : 12/20 = 0.600 CI [0.387, 0.781]
OVER-ABSTENTION  (gold 1 -> aggregate)    : 0/20 = 0.000 CI [0.0, 0.161]
GENERATION GUARD (gold 2/3/4 kept off it) : 45/60 = 0.750 CI [0.628, 0.842]

routing accuracy per gold bucket x language:
             acc   n
bucket lang         
1      de    1.0  10
       en    1.0  10
2      de    1.0  10
       en    0.9  10
3      de    0.9  10
       en    0.9  10
4      de    0.0  10
       en    0.8  10

misrouted questions, verbatim (15):
  [gold b2 en] routed retrieval (bucket 1)
    Q: Which importance level is most common among the judgments?
    trigger: no aggregate trigger -> retrieval
  [gold b3 en] routed retrieval (bucket 1)
    Q: Do married parents win contact d

In [1]:
# DISPATCH CONFUSION MATRIX -----------------------------------------------------------------
# 0.733 strict 4-way accuracy is a real number that exists nowhere else in the repo, but on its
# own it says nothing about WHICH capability is being lost. The matrix does.
BUCKETS = [1, 2, 3, 4]
cm = (pd.crosstab(gq.bucket, gq.pred)
        .reindex(index=BUCKETS, columns=BUCKETS, fill_value=0).fillna(0).astype(int))
cm.index.name, cm.columns.name = "gold", "predicted"
print("predicted (columns) vs gold (rows) -- dispatch bucket")
print(cm.to_string())
print("\n1 = retrieval + grounded generation | 2 = NL->SQL over metadata")
print("3 = content aggregate (extracted field, or refusal) | 4 = diachronic keyness")

g3p1 = int(cm.loc[3, 1])
g4p1 = int(cm.loc[4, 1])
g2p4, g3p4 = int(cm.loc[2, 4]), int(cm.loc[3, 4])
print(f"\ncells that matter for faithfulness:")
print(f"  gold 3 -> predicted 1 : {g3p1}  a corpus-wide claim generated from ~6 passages")
print(f"  gold 4 -> predicted 1 : {g4p1}  a corpus-wide CHANGE narrated from ~6 passages")
print(f"  gold 2 -> predicted 4 : {g2p4}  a metadata count answered as a wording-change story")
print(f"  gold 3 -> predicted 4 : {g3p4}  a content count answered as a wording-change story")
if int((gq.bucket == 4).sum()) == 0:
    print("  (the gold set carries no bucket-4 question, so the gold-4 row is empty by "
          "construction -- the diachronic path is exercised only as a false positive here)")

reg("dispatch_matrix", "Dispatch confusion matrix (4x4)",
    "When the dispatcher is wrong, WHICH capability does the question end up in -- and is that "
    "failure a fabrication risk or a lost convenience?",
    f"Full predicted-vs-gold bucket matrix over the same {n} frozen gold questions, from the "
    f"same deployed decision functions as the routing metric. Buckets 1-4, all four populated "
    f"({int((gq.bucket == 4).sum())} gold bucket-4 items), so every off-diagonal cell is a "
    f"failure the gold set can actually observe.",
    {"n_questions": n,
     "matrix": {f"gold{b}": {f"pred{p}": int(cm.loc[b, p]) for p in BUCKETS} for b in BUCKETS},
     "strict_4-way_accuracy": pct(bucket_k / n),
     "strict_4-way_accuracy_ci95": ci(bucket_k, n),
     "gold3_predicted1_corpus_claim_from_passages": g3p1,
     "gold4_predicted1_corpus_change_narrated_from_passages": g4p1,
     "gold2_predicted4": g2p4, "gold3_predicted4": g3p4,
     "gold3_predicted2_content_question_to_nl2sql": int(cm.loc[3, 2]),
     "gold3_at_nl2sql_refused_by_its_own_denylist": b3_denied,
     "gold_bucket4_questions": int((gq.bucket == 4).sum())},
    "gold 3 predicted 1 generates a corpus-wide claim from six passages; gold 3 predicted 2 "
    "hands a content question to a translator that can only see metadata columns",
    "gold",
    f"Strict 4-way accuracy {pct(bucket_k / n)} (CI {ci(bucket_k, n)}). {g3p1 + g4p1} "
    f"question(s) sit in the two cells that fabricate from retrieval -- {g3p1} a statistic "
    f"(gold 3 -> predicted 1) and {g4p1} a change narrative (gold 4 -> predicted 1). The other "
    f"cell is gold 3 -> predicted 2, {int(cm.loc[3, 2])} content-aggregate questions handed to "
    f"the NL->SQL translator, which is what the retired alienation branch leaves behind "
    f"(deployed fields: {sorted(DEPLOYED_FIELDS) or 'none'}): {b3_denied} of them are refused "
    f"by nl2sql()'s own deny-list "
    f"before any model call, and the fate of the other {int(cm.loc[3, 2]) - b3_denied} is not "
    f"measured by this metric. The generation-guard number in the routing metric is the "
    f"load-bearing one; this matrix says what the remaining error is made of.",
    corpus_dependent=False,
    headline=("strict_4-way_accuracy", pct(bucket_k / n)))


predicted (columns) vs gold (rows) -- dispatch bucket
predicted   1   2  3  4
gold                   
1          20   0  0  0
2           1  19  0  0
3           2  11  7  0
4          12   0  0  8

1 = retrieval + grounded generation | 2 = NL->SQL over metadata
3 = content aggregate (extracted field, or refusal) | 4 = diachronic keyness

cells that matter for faithfulness:
  gold 3 -> predicted 1 : 2  a corpus-wide claim generated from ~6 passages
  gold 4 -> predicted 1 : 12  a corpus-wide CHANGE narrated from ~6 passages
  gold 2 -> predicted 4 : 0  a metadata count answered as a wording-change story
  gold 3 -> predicted 4 : 0  a content count answered as a wording-change story
2. DISPATCH CONFUSION MATRIX (4X4)
  QUESTION IT ANSWERS : When the dispatcher is wrong, WHICH capability does the question end up in -- and is that failure a fabrication risk or a lost convenience?
  HOW MEASURED        : Full predicted-vs-gold bucket matrix over the same 80 frozen gold questions, from the 

In [1]:
# CALIBRATION -------------------------------------------------------------------------------
# Both figures are computed HERE, at BOTH bin counts, on the same rows. The previous version
# compared a locally computed 10-bin raw ECE against a 5-bin calibrated constant copied from
# extraction_validation.ipynb and presented the pair as a before/after.
CAL_BINS = (5, 10)
_gold_raw = pd.read_csv(DATA / "echr_labeled_sample.csv")[["id", "gold_alienation_alleged"]]
_extf = pd.read_parquet(DATA / "echr_extracted.parquet")
mcal = _gold_raw.merge(_extf[["id", "alienation_conf", "alienation_conf_cal"]], on="id",
                       how="inner")
mcal["gold"] = pd.to_numeric(mcal.gold_alienation_alleged, errors="coerce")
mcal = mcal[mcal.gold.isin([0, 1])].copy()
y_cal = mcal.gold.astype(int).to_numpy()
raw = mcal.alienation_conf.to_numpy(float)
dep = mcal.alienation_conf_cal.to_numpy(float)     # deployed isotonic -- FIT ON THESE ROWS
oof = oof_isotonic(raw, y_cal, n_splits=5, random_state=42)
assert oof is not None, "labelled sample cannot support 5-fold stratified CV"

cal_vals = {"n": int(len(mcal)),
            "field": "alienation_alleged (transparent rules column, deployed 2026-09-18)",
            "protocol": "StratifiedKFold(5, shuffle=True, random_state=42) + "
                        "IsotonicRegression(out_of_bounds='clip', y_min=0, y_max=1)"}
cal_vals["n_bins"] = list(CAL_BINS)
for nb_ in CAL_BINS:
    cal_vals[f"ece_raw@{nb_}bins"] = pct(ece(raw, y_cal, nb_))
    cal_vals[f"ece_calibrated_out_of_fold@{nb_}bins"] = pct(ece(oof, y_cal, nb_))
cal_vals["brier_raw"] = pct(float(np.mean((raw - y_cal) ** 2)))
cal_vals["brier_calibrated_out_of_fold"] = pct(float(np.mean((oof - y_cal) ** 2)))
cal_vals["brier_calibrated_in_sample"] = pct(float(np.mean((dep - y_cal) ** 2)))
cal_vals["distinct_raw_confidence_values"] = int(pd.Series(raw).nunique())

for nb_ in CAL_BINS:
    print(f"n_bins={nb_:2d}: ECE raw {cal_vals[f'ece_raw@{nb_}bins']:.3f} -> "
          f"out-of-fold calibrated {cal_vals[f'ece_calibrated_out_of_fold@{nb_}bins']:.3f}")
print(f"Brier raw {cal_vals['brier_raw']} | out-of-fold {cal_vals['brier_calibrated_out_of_fold']} "
      f"| in-sample {cal_vals['brier_calibrated_in_sample']} (optimistic: the deployed isotonic "
      f"was fitted on these same rows)")
_spread = abs(cal_vals["ece_raw@5bins"] - cal_vals["ece_raw@10bins"])

reg("calibration", "Confidence calibration",
    "Are the confidence scores honest -- does a stated confidence of 0.8 really mean about 80% "
    "correct on cases the calibrator never saw?",
    "Expected Calibration Error on 120 frozen hand-labelled ECHR cases, raw vs out-of-fold "
    "isotonic, both computed in THIS notebook on the SAME rows and reported at BOTH 5 and 10 "
    "bins (eval_metrics.ece, where n_bins is a required argument). Out-of-fold protocol: "
    "StratifiedKFold(5, shuffle, seed 42); isotonic fitted on the training folds only, so every "
    "row is scored by a mapping that never saw its label. Bin-count sensitivity is shown rather "
    "than hidden: Nixon et al. (2019) demonstrate that metric variants reorder the same "
    "predictions.",
    cal_vals,
    "a cell reported at calibrated confidence 0.8 is right about 80% of the time on rows the "
    "isotonic fit excluded",
    "gold",
    f"Out-of-fold calibration improves ECE at both bin counts "
    f"({cal_vals['ece_raw@5bins']} -> {cal_vals['ece_calibrated_out_of_fold@5bins']} at 5 bins; "
    f"{cal_vals['ece_raw@10bins']} -> {cal_vals['ece_calibrated_out_of_fold@10bins']} at 10), "
    f"and the raw figure alone moves by {_spread:.3f} between the two bin counts -- which is "
    f"why the estimator is printed with the number. The Brier score is given out-of-fold and "
    f"in-sample side by side; only the out-of-fold pair supports an unseen-data claim. The whole "
    f"estimate rests on {cal_vals['distinct_raw_confidence_values']} distinct confidence values "
    f"(see the confidence-granularity metric), so it describes that many confidence levels "
    f"rather than a continuous probability. Measured "
    f"on the alienation_alleged column -- the rules field, the only one with reviewed "
    f"labels rather than model-adjudicated ones. It was re-deployed as a queryable "
    f"content field on 2026-09-18, so this now validates the CALIBRATION PROTOCOL *and* "
    f"the confidence of a column the system answers from. Note the deployed column refits\n"
    f"isotonic on a different target -- P(cell is correct) rather than P(allegation) -- "
    f"so the ECE here describes the protocol, not the deployed threshold.",
    corpus_dependent=False,
    headline=("ece_calibrated_out_of_fold@10bins", cal_vals["ece_calibrated_out_of_fold@10bins"]))


n_bins= 5: ECE raw 0.179 -> out-of-fold calibrated 0.068
n_bins=10: ECE raw 0.203 -> out-of-fold calibrated 0.084
Brier raw 0.183 | out-of-fold 0.151 | in-sample 0.138 (optimistic: the deployed isotonic was fitted on these same rows)
3. CONFIDENCE CALIBRATION
  QUESTION IT ANSWERS : Are the confidence scores honest -- does a stated confidence of 0.8 really mean about 80% correct on cases the calibrator never saw?
  HOW MEASURED        : Expected Calibration Error on 120 frozen hand-labelled ECHR cases, raw vs out-of-fold isotonic, both computed in THIS notebook on the SAME rows and reported at BOTH 5 and 10 bins (eval_metrics.ece, where n_bins is a required argument). Out-of-fold protocol: StratifiedKFold(5, shuffle, seed 42); isotonic fitted on the training folds only, so every row is scored by a mapping that never saw its label. Bin-count sensitivity is shown rather than hidden: Nixon et al. (2019) demonstrate that metric variants reorder the same predictions.
  CORPUS-DEPENDENT    :

In [1]:
# CONFIDENCE GRANULARITY -- what the ECE is actually binning ---------------------------------
# A calibration error is a statement about a probability. The extractor's "confidence" is not
# one: it is an arithmetic score over mention counts, so it takes a SMALL NUMBER OF DISCRETE
# VALUES and every bin is dominated by a few atoms. Binning finer than the score's own
# granularity invents resolution that is not in the data, which is the precondition for reading
# the calibration metric at all -- so it is measured rather than assumed.
vals = pd.Series(raw).value_counts().sort_index()
distinct_raw = int(vals.size)
distinct_oof = int(pd.Series(np.round(oof, 6)).nunique())
top_share = float(vals.max() / len(raw))
occupied = {}
for nb_ in CAL_BINS:
    edges = np.linspace(0, 1, nb_ + 1)
    idx = np.clip(np.digitize(raw, edges) - 1, 0, nb_ - 1)
    occupied[nb_] = int(pd.Series(idx).nunique())
print(f"distinct raw confidence values: {distinct_raw} over n={len(raw)} cases")
print(f"  most common value {vals.idxmax()} carries {int(vals.max())}/{len(raw)} "
      f"= {top_share:.3f} of the sample")
print(f"  distinct out-of-fold calibrated values: {distinct_oof}")
print(f"    a SINGLE isotonic fit is a monotone step function and can only merge values; "
      f"out-of-fold uses five fold-specific fits, so one raw value can map to several "
      f"calibrated ones and the count can exceed {distinct_raw}. That is variance across folds, "
      f"not resolution the score possesses.")
for nb_, occ in occupied.items():
    print(f"  n_bins={nb_:2d}: {occ}/{nb_} bins occupied")
print("  value -> count:", {round(float(k), 3): int(v) for k, v in vals.items()})

reg("confidence_granularity", "Confidence granularity behind the ECE",
    "How many distinct confidence values does the calibration error actually rest on -- i.e. is "
    "there enough resolution in the score for a binned estimator to mean anything?",
    f"Count the distinct self-reported confidence values in the same {len(raw)} labelled cases "
    f"the calibration metric uses, the support behind each, and how many of the {list(CAL_BINS)} "
    f"bins are occupied. No LLM. This is a precondition check, not a quality score: it says what "
    f"the ECE can and cannot resolve.",
    {"n": int(len(raw)),
     "distinct_raw_confidence_values": distinct_raw,
     "distinct_out_of_fold_calibrated_values": distinct_oof,
     "oof_value_count_note": ("out-of-fold uses five fold-specific isotonic fits, so this count "
                              "can exceed the raw count -- it is fold-to-fold variance, not "
                              "added resolution; a single fit can only merge values"),
     "most_common_value": pct(float(vals.idxmax())),
     "most_common_value_support": int(vals.max()),
     "most_common_value_share": pct(top_share),
     "bins_occupied": {f"of_{k}": v for k, v in occupied.items()},
     "value_support": {str(round(float(k), 3)): int(v) for k, v in vals.items()},
     "field": "alienation_alleged (transparent rules column, deployed 2026-09-18)"},
    f"{distinct_raw} distinct values, one of which ({pct(float(vals.idxmax()))}) covers "
    f"{int(vals.max())} of {len(raw)} cases",
    "gold",
    f"The ECE rests on only {distinct_raw} distinct confidence values across {len(raw)} cases, "
    f"and a single value ({pct(float(vals.idxmax()))}) accounts for {pct(top_share*100, 1)}% of "
    f"them. At 10 bins {occupied[10]} are occupied and at 5 bins {occupied[5]} are -- so the "
    f"finer estimator adds no resolution the score itself possesses, which is the concrete form "
    f"the bin-sensitivity in the calibration metric takes. Read the ECE as a statement about "
    f"{distinct_raw} confidence levels, not about a continuous probability.",
    corpus_dependent=False,
    headline=("distinct_raw_confidence_values", distinct_raw))


distinct raw confidence values: 14 over n=120 cases
  most common value 0.02 carries 37/120 = 0.308 of the sample
  distinct out-of-fold calibrated values: 16
    a SINGLE isotonic fit is a monotone step function and can only merge values; out-of-fold uses five fold-specific fits, so one raw value can map to several calibrated ones and the count can exceed 14. That is variance across folds, not resolution the score possesses.
  n_bins= 5: 5/5 bins occupied
  n_bins=10: 8/10 bins occupied
  value -> count: {0.02: 37, 0.05: 6, 0.12: 16, 0.15: 2, 0.17: 1, 0.27: 1, 0.3: 11, 0.4: 6, 0.42: 5, 0.52: 11, 0.65: 6, 0.75: 1, 0.77: 7, 0.87: 10}
4. CONFIDENCE GRANULARITY BEHIND THE ECE
  QUESTION IT ANSWERS : How many distinct confidence values does the calibration error actually rest on -- i.e. is there enough resolution in the score for a binned estimator to mean anything?
  HOW MEASURED        : Count the distinct self-reported confidence values in the same 120 labelled cases the calibration met

In [1]:
# RETRIEVAL QUALITY + STALENESS CHECK -------------------------------------------------------
# retrieval_judgments.csv judges the top-6 of the PRE-SEPTEMBER index. Booting the current index
# and scoring those labels describes a ranking the system no longer produces, so the ranking is
# re-run and the overlap measured before any number is reported.
rj = pd.read_csv(DATA / "retrieval_judgments.csv")
rj = rj[pd.to_numeric(rj.relevant, errors="coerce").notna()].copy()
rj["relevant"] = rj.relevant.astype(int)
_depth = rj.groupby("qid").size()
JUDGED_K = int(_depth.max())          # 6 = EVAL_K in retrieval_evaluation.ipynb
STALE_MIN_OVERLAP = 0.90


def per_q(g):
    g = g.sort_values("rank")
    rel = g.relevant.to_numpy()
    k = min(JUDGED_K, len(rel))
    hits = np.where(rel == 1)[0]
    dcg = sum(rel[i] / math.log2(i + 2) for i in range(k))
    ideal = sorted(rel, reverse=True)
    idcg = sum(ideal[i] / math.log2(i + 2) for i in range(k))
    return pd.Series({"p": float(rel[:k].mean()) if k else 0.0,
                      "rr": 1.0 / (hits[0] + 1) if len(hits) else 0.0,
                      "ndcg": dcg / idcg if idcg else 0.0})


ag = rj.groupby("qid")[["rank", "relevant"]].apply(per_q)

# --- is the frozen judgment set still describing this index? ---
judged_ids = rj.groupby("qid").chunk_id.apply(set).to_dict()
qset = rj.drop_duplicates("qid")[["qid", "query_lang", "query"]]
print(f"re-running retrieval for {len(qset)} frozen queries against the current "
      f"{N_INDEX}-passage index ...")
_ov = []
_t0 = time.time()
for r in qset.itertuples():
    # same switches retrieval_evaluation.ipynb used to build the judged pool: this measures the
    # ranker itself, so the similarity cut and the per-case cap stay off
    now = [h["chunk_id"] for h in retrieve(r.query, k=JUDGED_K, score_floor=None,
                                           score_margin=None, max_per_case=None)]
    _ov.append({"qid": r.qid, "lang": r.query_lang, "n_now": len(now),
                "overlap": sum(c in judged_ids[r.qid] for c in now) / max(1, len(now))})
ov = pd.DataFrame(_ov)
judged_overlap = float(ov.overlap.mean())
STALE = judged_overlap < STALE_MIN_OVERLAP
print(f"  done in {time.time()-_t0:.0f}s | mean overlap of the current top-{JUDGED_K} with the "
      f"judged pool: {judged_overlap:.3f} | queries with zero overlap: "
      f"{int((ov.overlap == 0).sum())}/{len(ov)}")
if STALE:
    print(f"  !! STALE: below {STALE_MIN_OVERLAP:.2f}. The numbers below stand for the index "
          f"they were judged on and MUST be labelled with it.")
    print(f"  !! To refresh: run retrieval_evaluation.ipynb to regenerate "
          f"data/retrieval_judgments_template.csv against the current index, re-judge, save as "
          f"data/retrieval_judgments.csv. Old labels cannot be reused -- the retrieved chunk "
          f"ids differ, so they do not join.")

retr_vals = {"n_queries": int(ag.shape[0]), "judged_depth_k": JUDGED_K,
             f"P@{JUDGED_K}": pct(float(ag.p.mean())),
             "MRR": pct(float(ag.rr.mean())),
             f"nDCG@{JUDGED_K} (pool-limited)": pct(float(ag.ndcg.mean())),
             "n_judged_hits": int(len(rj)),
             "judged_on_index": "pre-September-2026 re-import",
             "current_index_passages": N_INDEX,
             "current_top6_overlap_with_judged_pool": pct(judged_overlap),
             "queries_with_zero_overlap": int((ov.overlap == 0).sum()),
             "stale_threshold": STALE_MIN_OVERLAP}

reg("retrieval", "Retrieval quality",
    "Does the retriever put genuinely relevant passages at the top -- the evidence every "
    "grounded answer is built on?",
    f"Frozen query->passage relevance judgments (24 queries x top-{JUDGED_K}, strict criterion: "
    f"relevant only if the chunk states the deciding court's own rule). Standard IR metrics at "
    f"k={JUDGED_K}, which is the judged pool depth (EVAL_K in retrieval_evaluation.ipynb) -- the "
    f"same k the retrieval report and the thesis chapters use. nDCG is POOL-LIMITED: the ideal "
    f"DCG is computed from the {JUDGED_K} judged rows, not from the full corpus, so it is not "
    f"comparable with published nDCG figures. P@{JUDGED_K} and MRR are means over QUERIES, not "
    f"hit-level proportions, so no binomial interval is given for them -- the unit is the "
    f"query. Staleness is checked, not assumed: retrieval is re-run for every frozen query and "
    f"the overlap with the judged pool is reported.",
    retr_vals,
    f"across {int(ag.shape[0])} judged queries the first relevant passage is usually within the "
    f"top two hits",
    "gold",
    (f"P@{JUDGED_K} {retr_vals[f'P@{JUDGED_K}']}, MRR {retr_vals['MRR']}, "
     f"nDCG@{JUDGED_K} {retr_vals[f'nDCG@{JUDGED_K} (pool-limited)']} -- but only "
     f"{pct(judged_overlap*100, 1)}% of what the current index returns for these queries was "
     f"ever judged, so these figures describe the PRE-RE-IMPORT ranking and are labelled with "
     f"it. They cannot be cited as properties of the deployed system until the judgment template "
     f"is regenerated and re-judged."
     if STALE else
     f"P@{JUDGED_K} {retr_vals[f'P@{JUDGED_K}']}, MRR {retr_vals['MRR']}, "
     f"nDCG@{JUDGED_K} {retr_vals[f'nDCG@{JUDGED_K} (pool-limited)']} over "
     f"{int(ag.shape[0])} queries, and {pct(judged_overlap*100, 1)}% of the current top-"
     f"{JUDGED_K} is still inside the judged pool, so the labels still describe this index."),
    corpus_dependent=True, status="STALE" if STALE else "computed",
    headline=(f"P@{JUDGED_K}", retr_vals[f"P@{JUDGED_K}"]))


re-running retrieval for 24 frozen queries against the current 155164-passage index ...
  done in 15s | mean overlap of the current top-6 with the judged pool: 0.368 | queries with zero overlap: 4/24
  !! STALE: below 0.90. The numbers below stand for the index they were judged on and MUST be labelled with it.
  !! To refresh: run retrieval_evaluation.ipynb to regenerate data/retrieval_judgments_template.csv against the current index, re-judge, save as data/retrieval_judgments.csv. Old labels cannot be reused -- the retrieved chunk ids differ, so they do not join.
5. RETRIEVAL QUALITY   [STALE]
  QUESTION IT ANSWERS : Does the retriever put genuinely relevant passages at the top -- the evidence every grounded answer is built on?
  HOW MEASURED        : Frozen query->passage relevance judgments (24 queries x top-6, strict criterion: relevant only if the chunk states the deciding court's own rule). Standard IR metrics at k=6, which is the judged pool depth (EVAL_K in retrieval_evaluation

In [1]:
# RETRIEVAL SUBGROUPS -----------------------------------------------------------------------
# One P@k figure hides the two retrieval findings the chapter actually needs: whether a German
# query reaches the German corpora, and whether an ECHR hit landed in the Court's own assessment.
def _grp(frame, by):
    g = frame.groupby(by).relevant.agg(["mean", "size"])
    return {str(k): {"precision": pct(v["mean"]), "n": int(v["size"])} for k, v in g.iterrows()}


by_lang = _grp(rj, "query_lang")
by_juris = _grp(rj, "jurisdiction")
echr = rj[rj.jurisdiction == "ECHR"]
by_section = _grp(echr, "section") if len(echr) else {}
law_share = float((echr.section == "LAW").mean()) if len(echr) else float("nan")
law_p = float(echr[echr.section == "LAW"].relevant.mean()) if (echr.section == "LAW").any() else None
nonlaw = echr[echr.section != "LAW"]
nonlaw_p = float(nonlaw.relevant.mean()) if len(nonlaw) else None

print("precision by query language :", by_lang)
print("precision by jurisdiction   :", by_juris)
print("precision by ECHR section   :", by_section)
print(f"LAW share of ECHR hits      : {law_share:.3f}  "
      f"(LAW precision {pct(law_p)} vs non-LAW {pct(nonlaw_p)})")

sub_vals = {"n_judged_hits": int(len(rj)),
            "precision_by_query_language": by_lang,
            "precision_by_jurisdiction": by_juris,
            "precision_by_echr_section": by_section,
            "law_share_of_echr_hits": pct(law_share),
            "echr_law_precision": pct(law_p), "echr_non_law_precision": pct(nonlaw_p),
            "n_echr_hits": int(len(echr)),
            "judged_on_index": "pre-September-2026 re-import",
            "current_top6_overlap_with_judged_pool": pct(judged_overlap)}

_gap = (None if law_p is None or nonlaw_p is None else pct(law_p - nonlaw_p))
reg("retrieval_subgroups", "Retrieval precision by language, jurisdiction and ECHR section",
    "Is retrieval quality uniform -- or does it depend on the language of the question and on "
    "which part of a judgment the hit landed in?",
    "The same frozen judged hits as the retrieval-quality metric, broken down by query language (de/en), by the "
    "jurisdiction of the retrieved chunk (ECHR / AT (OGH) / CH) and, for ECHR hits, by document "
    "section (LAW / FACTS / HEADER / PROCEDURE / OPERATIVE). Cell sizes are reported with every "
    "rate, because some sections carry very few judged hits.",
    sub_vals,
    "an ECHR hit in the LAW section is the Court's own assessment; a FACTS or HEADER hit is a "
    "recital, topically close and legally inert",
    "gold",
    (f"ECHR precision is a GENRE effect: LAW-section hits score {pct(law_p)} "
     f"(n={int((echr.section == 'LAW').sum())}) against {pct(nonlaw_p)} (n={len(nonlaw)}) for "
     f"everything else"
     + (f" (a gap of {_gap})" if _gap is not None else "")
     + f", and only {pct(law_share*100, 1)}% of ECHR hits land in LAW. The per-language and "
       f"per-jurisdiction splits are in the result dict with their cell sizes. All of it is "
       f"judged on the PRE-RE-IMPORT ranking (see the retrieval-quality metric), so it is a finding about the retriever's "
       f"failure mode, not a current operating figure."
     if law_p is not None and nonlaw_p is not None else
     "Subgroup precisions are in the result dict with their cell sizes; the ECHR section split "
     "could not be computed because no ECHR hits were judged."),
    corpus_dependent=True, status="STALE" if STALE else "computed",
    headline=("law_share_of_echr_hits", pct(law_share)))


precision by query language : {'de': {'precision': 0.594, 'n': 96}, 'en': {'precision': 0.167, 'n': 48}}
precision by jurisdiction   : {'AT (OGH)': {'precision': 0.606, 'n': 33}, 'CH': {'precision': 0.587, 'n': 63}, 'ECHR': {'precision': 0.167, 'n': 48}}
precision by ECHR section   : {'FACTS': {'precision': 0.0, 'n': 28}, 'HEADER': {'precision': 0.0, 'n': 7}, 'LAW': {'precision': 0.727, 'n': 11}, 'OPERATIVE': {'precision': 0.0, 'n': 1}, 'PROCEDURE': {'precision': 0.0, 'n': 1}}
LAW share of ECHR hits      : 0.229  (LAW precision 0.727 vs non-LAW 0.0)
6. RETRIEVAL PRECISION BY LANGUAGE, JURISDICTION AND ECHR SECTION   [STALE]
  QUESTION IT ANSWERS : Is retrieval quality uniform -- or does it depend on the language of the question and on which part of a judgment the hit landed in?
  HOW MEASURED        : The same frozen judged hits as the retrieval-quality metric, broken down by query language (de/en), by the jurisdiction of the retrieved chunk (ECHR / AT (OGH) / CH) and, for ECHR hits, b

### Content fields (`field_factory` → `field_deploy`)

Written against **any** field the factory ships, discovered from
`data/field_<name>_labels.csv` and its sidecar `data/field_<name>_meta.json`. No field name and
no lexicon is hardcoded, and the retired `alienation_alleged` column is deliberately not wired
in. Every scorable field is measured; the **primary** field — the one that headlines these three
metrics — is the *deployed* one, because that is the column a Bucket-3 answer is actually
computed from. `EVAL_FIELD=<name>` overrides the choice. A field that cannot be scored (numeric
rather than boolean, or not yet adjudicated) is listed **with the reason** instead of dropped,
and if no field is scorable all three register as pending.

**Who wrote the gold matters as much as the F1.** Each field's `labels_provenance` is carried
through from its sidecar into every result dict below, because an F1 against labels a *second
model* adjudicated is agreement between two models — not a human validation, however high the
flip rate.


In [1]:
# FIELD DISCOVERY ------------------------------------------------------------------------------
def discover_fields():
    """Every field the factory has produced reviewed labels for, with its sidecar meta and
    deployed column if it has them.

    Parameterised by name, never by column: field_deploy.ipynb writes
    data/field_<name>_labels.csv (id, detector_conf, draft_label|draft_value, draft_conf,
    gold_<name>), data/field_<name>_meta.json (definition, lexicon, threshold, validation,
    labels provenance) and data/field_<name>_deployed.parquet. Anything the factory ships is
    measurable here with no code change. A field that cannot be scored is returned with the
    REASON attached rather than dropped, so it stays visible.
    """
    out = []
    for p in sorted(DATA.glob("field_*_labels.csv")):
        name = p.name[len("field_"):-len("_labels.csv")]
        meta_p, dep_p = DATA / f"field_{name}_meta.json", DATA / f"field_{name}_deployed.parquet"
        meta = json.loads(meta_p.read_text()) if meta_p.exists() else None
        e = {"name": name, "meta": meta, "labels": None, "skip": None, "n_labelled": 0,
             "deployed": pd.read_parquet(dep_p) if dep_p.exists() else None,
             "is_deployed": bool(dep_p.exists() and meta is not None)}
        lab = pd.read_csv(p)
        gcol = f"gold_{name}"
        if gcol not in lab.columns:
            e["skip"] = f"labels file carries no {gcol} column"
            out.append(e); continue
        lab["gold"] = pd.to_numeric(lab[gcol], errors="coerce")
        lab = lab[lab.gold.notna()].copy()
        e["n_labelled"] = int(len(lab))
        if lab.empty:
            e["skip"] = "template exists but no row carries a reviewed label yet"
            out.append(e); continue
        kind = (meta or {}).get("kind") or ("boolean" if "draft_label" in lab.columns
                                            else "non-boolean")
        e["kind"] = kind
        if kind != "boolean" or "draft_label" not in lab.columns:
            # a numeric field (draft_value) is a different measurement problem -- thresholded
            # P/R/F1 would be a category error, so it is named and skipped, not coerced
            e["skip"] = (f"field is {kind} (draft_value, not draft_label); these blocks measure "
                         f"BINARY classification and will not coerce a numeric field into one")
            out.append(e); continue
        lab["gold"] = lab.gold.astype(int)
        e["labels"] = lab
        out.append(e)
    return out


def field_provenance(f):
    """WHERE the gold labels came from. This decides what an F1 against them means."""
    return ((f.get("meta") or {}).get("labels_provenance")
            or "not recorded (no field_<name>_meta.json sidecar)")


def field_adjudicator(f):
    """'model', 'human' or 'unknown' -- read from the sidecar, never assumed."""
    p = field_provenance(f).lower()
    if "model-adjudicated" in p or "not a human" in p:
        return "model"
    if "human" in p or "review" in p:
        return "human"
    return "unknown"


def field_model(f):
    return (f.get("meta") or {}).get("model") or "not recorded"


FIELDS = discover_fields()
SCORABLE = [f for f in FIELDS if f["labels"] is not None]
_want = os.environ.get("EVAL_FIELD")
# The PRIMARY field is the DEPLOYED one: that is the column a Bucket-3 answer is actually
# computed from. A field with labels but no deployed parquet is measured too, but it answers
# no question yet.
PRIMARY = (next((f for f in SCORABLE if f["name"] == _want), None)
           or next((f for f in SCORABLE if f["is_deployed"]), None)
           or (SCORABLE[0] if SCORABLE else None))
_tmpl = sorted(p.name for p in DATA.glob("field_*_template.csv"))
FIELD_PENDING = (
    "no field has reviewed labels yet"
    + (f" (drafted templates awaiting review: {', '.join(_tmpl)})" if _tmpl else "")
    + ". The factory drafts labels and they are then adjudicated; until that is saved as "
      "data/field_<name>_labels.csv there is nothing to score.")
_UNBLOCK = ("adjudicate a drafted template into data/field_<name>_labels.csv, then run "
            "field_deploy.ipynb")

for f in FIELDS:
    tag = ("DEPLOYED" if f["is_deployed"] else "labelled only") if not f["skip"] else "NOT SCORED"
    print(f"  {f['name']:24s} {tag:14s} n={f['n_labelled']:4d} | gold: {field_provenance(f)[:72]}")
    if f["skip"]:
        print(f"  {'':24s} -> {f['skip']}")
if PRIMARY:
    print(f"\nprimary field (headlines the three field metrics): {PRIMARY['name']} "
          f"({'deployed' if PRIMARY['is_deployed'] else 'labelled but NOT deployed'}); "
          f"{len(SCORABLE)} scorable of {len(FIELDS)} labelled fields")
    print(f"deployed content fields the dispatcher can route to: {sorted(DEPLOYED_FIELDS) or 'none'}")
else:
    print("content-field metrics: PENDING --", FIELD_PENDING)


  applicant_is_father      DEPLOYED       n= 120 | gold: model-adjudicated gold (claude-opus-5), read from the case evidence BLIN
  child_age                NOT SCORED     n=  49 | gold: not recorded (no field_<name>_meta.json sidecar)
                           -> field is non-boolean (draft_value, not draft_label); these blocks measure BINARY classification and will not coerce a numeric field into one
  child_heard              DEPLOYED       n= 113 | gold: model-adjudicated gold (claude-opus-5), read from the case evidence BLIN
  coercive_measures        DEPLOYED       n= 120 | gold: model-adjudicated gold (claude-opus-5), read from the case evidence BLIN
  expert_opinion_ordered   DEPLOYED       n= 120 | gold: model-adjudicated gold (claude-opus-5), read from the case evidence BLIN

primary field (headlines the three field metrics): applicant_is_father (deployed); 4 scorable of 5 labelled fields
deployed content fields the dispatcher can route to: ['alienation_alleged', 'applicant_

In [1]:
# CONTENT-FIELD CLASSIFICATION ---------------------------------------------------------------
def score_field(f):
    """P/R/F1 + count bias for every extractor run against one field's reviewed labels."""
    fl, fname = f["labels"], f["name"]
    y = fl.gold.to_numpy(int)
    ex = {}
    if "draft_label" in fl.columns:
        ex["llm_drafts"] = fl.draft_label.astype(int).to_numpy()
    if "detector_conf" in fl.columns:
        ex["lexicon_detector@0.5"] = (fl.detector_conf >= 0.5).astype(int).to_numpy()
    if f["deployed"] is not None and f["meta"]:
        thr = f["meta"]["threshold"]
        dm = f["deployed"].set_index("id")
        col, cal = dm[fname], dm["conf_cal"]
        ex[f"deployed@conf_cal>={thr}"] = fl.id.map(
            lambda i: bool(col.get(i, False)) and float(cal.get(i, 0.0)) >= thr
        ).astype(int).to_numpy()
    per_ex, bias = {}, {}
    for label, pred in ex.items():
        s = prf1(y, pred)
        s["precision_ci95"] = ci(s["tp"], s["tp"] + s["fp"]) if (s["tp"] + s["fp"]) else None
        s["recall_ci95"] = ci(s["tp"], s["tp"] + s["fn"]) if (s["tp"] + s["fn"]) else None
        per_ex[label] = s
        bias[label] = count_bias(s["tp"], s["fp"], s["fn"])
    agree = None
    if len(ex) >= 2:
        (na, pa), (nb_, pb) = list(ex.items())[:2]
        oka, okb = pa == y, pb == y
        agree = {"extractor_A": na, "extractor_B": nb_,
                 "both_right": int((oka & okb).sum()), "only_A_right": int((oka & ~okb).sum()),
                 "only_B_right": int((~oka & okb).sum()), "both_wrong": int((~oka & ~okb).sum())}
    best = max(per_ex, key=lambda k: per_ex[k]["f1"])
    return {"n_labels": int(len(fl)), "gold_positive_rate": pct(float(y.mean())),
            "is_deployed": f["is_deployed"], "labels_provenance": field_provenance(f),
            "adjudicator": field_adjudicator(f), "drafting_model": field_model(f),
            "best_extractor": best, "best_extractor_f1": per_ex[best]["f1"],
            "best_extractor_count_ratio": bias[best]["count_ratio"],
            "per_extractor": per_ex, "count_bias": bias, "agreement": agree}


if PRIMARY is None:
    reg_pending("field_classification",
                "Content-field classification (precision / recall / F1 + count bias)",
                "How good is a deployed content field at the thing a Bucket-3 answer claims -- "
                "and is its error budget balanced enough for the COUNT it reports to be usable?",
                "TP/FP/FN/TN, precision, recall and F1 for every extractor run against the same "
                "reviewed labels, plus the count bias (FP - FN) that decides whether an "
                "aggregate answer over the field is usable. Parameterised by field name.",
                FIELD_PENDING, "gold", corpus_dependent=False, unblocks=_UNBLOCK)
else:
    per_field = {f["name"]: score_field(f) for f in SCORABLE}
    for name, s in per_field.items():
        mark = " [DEPLOYED]" if s["is_deployed"] else ""
        print(f"  {name}{mark}  n={s['n_labels']} gold+={s['gold_positive_rate']} "
              f"({s['adjudicator']}-adjudicated gold)")
        for label, e in s["per_extractor"].items():
            b = s["count_bias"][label]
            print(f"      {label:28s} P={e['precision']:.3f} R={e['recall']:.3f} "
                  f"F1={e['f1']:.3f} | predicted {b['predicted_count']} vs true "
                  f"{b['true_count']} (bias {b['count_bias']:+d}, ratio {b['count_ratio']})")
        if s["agreement"]:
            print(f"      agreement: {s['agreement']}")
    _skipped = {f["name"]: f["skip"] for f in FIELDS if f["skip"]}
    if _skipped:
        print("  not scored:", _skipped)

    def _best_counter(s):
        """The extractor whose reported count is closest to the truth (ratio nearest 1)."""
        return min(s["count_bias"],
                   key=lambda k: abs((s["count_bias"][k]["count_ratio"] or 0) - 1))

    # where ranking by F1 and ranking by count accuracy disagree, F1 would pick the extractor
    # that reports the worse number -- the concrete case for measuring both
    diverge = {n: {"best_by_f1": s["best_extractor"],
                   "its_count_ratio": s["count_bias"][s["best_extractor"]]["count_ratio"],
                   "best_by_count": _best_counter(s),
                   "its_count_ratio_": s["count_bias"][_best_counter(s)]["count_ratio"],
                   "its_f1": s["per_extractor"][_best_counter(s)]["f1"]}
               for n, s in per_field.items() if _best_counter(s) != s["best_extractor"]}
    if diverge:
        print("  F1 and count accuracy pick DIFFERENT extractors on:", list(diverge))
        for n, d in diverge.items():
            print(f"      {n}: best F1 `{d['best_by_f1']}` reports ratio {d['its_count_ratio']}, "
                  f"while `{d['best_by_count']}` (F1 {d['its_f1']}) reports ratio "
                  f"{d['its_count_ratio_']}")

    P = per_field[PRIMARY["name"]]
    _pb = P["count_bias"][P["best_extractor"]]
    reg("field_classification",
        f"Content-field classification -- `{PRIMARY['name']}` (P / R / F1 + count bias)",
        "How good is the deployed content field at the thing a Bucket-3 answer claims -- and is "
        "its error budget balanced enough for the COUNT it reports to be usable?",
        f"TP/FP/FN/TN, precision, recall and F1 for every extractor run against the same "
        f"reviewed labels, with Wilson intervals on precision and recall; plus count bias "
        f"(FP - FN), which nothing else in the repo computes and which matters more than F1 for "
        f"an aggregate answer. Computed for all {len(SCORABLE)} scorable fields and headlined by "
        f"`{PRIMARY['name']}`, the field the dispatcher can actually route to. Nothing is "
        f"hardcoded: fields, their labels and their sidecar meta are discovered. The provenance "
        f"of each field's gold labels is carried through, because it decides what an F1 against "
        f"them means.",
        {"primary_field": PRIMARY["name"],
         "primary_is_deployed": PRIMARY["is_deployed"],
         "best_extractor_f1": P["best_extractor_f1"],
         "best_extractor": P["best_extractor"],
         "labels_provenance": P["labels_provenance"],
         "adjudicator": P["adjudicator"],
         "fields_scored": len(SCORABLE),
         "fields_not_scored": {f["name"]: f["skip"] for f in FIELDS if f["skip"]} or None,
         "f1_and_count_accuracy_disagree_on": diverge or None,
         "per_field": per_field},
        f"`{PRIMARY['name']}` predicted {_pb['predicted_count']} cases against a true "
        f"{_pb['true_count']}",
        "gold",
        f"On `{PRIMARY['name']}` the best extractor (`{P['best_extractor']}`) scores F1 "
        f"{P['best_extractor_f1']} over {P['n_labels']} labels and reports "
        f"{_pb['predicted_count']} positives where the gold says {_pb['true_count']} "
        f"(bias {_pb['count_bias']:+d}, ratio {_pb['count_ratio']}) -- so the count it would "
        f"report is {'an undercount' if _pb['count_bias'] < 0 else 'an overcount' if _pb['count_bias'] > 0 else 'balanced'}, "
        f"which F1 alone does not say. "
        + (f"The gold is {P['adjudicator']}-adjudicated ({P['labels_provenance']}), so this F1 "
           f"is agreement between two models, not agreement with a human judgment."
           if P["adjudicator"] == "model" else
           f"Gold provenance: {P['labels_provenance']}.")
        + f" {len(SCORABLE)} field(s) scored; per-field numbers are in the result dict."
        + (f" On {len(diverge)} of them ({', '.join(diverge)}) ranking by F1 and ranking by count "
           f"accuracy pick DIFFERENT extractors -- e.g. on "
           f"`{list(diverge)[0]}` the best-F1 extractor reports a count ratio of "
           f"{list(diverge.values())[0]['its_count_ratio']} while another reaches "
           f"{list(diverge.values())[0]['its_count_ratio_']}. Choosing an extractor by F1 alone "
           f"would therefore choose the worse counter, which is the case for reporting both."
           if diverge else
           " On every field scored, the best-F1 extractor is also the one whose count is "
           "closest to the truth, so the two rankings agree here."),
        corpus_dependent=False,
        headline=("best_extractor_f1", P["best_extractor_f1"]),
        backend="ollama", model=field_model(PRIMARY))


  applicant_is_father [DEPLOYED]  n=120 gold+=0.65 (model-adjudicated gold)
      llm_drafts                   P=0.789 R=0.577 F1=0.667 | predicted 57 vs true 78 (bias -21, ratio 0.731)
      lexicon_detector@0.5         P=0.900 R=0.692 F1=0.783 | predicted 60 vs true 78 (bias -18, ratio 0.769)
      deployed@conf_cal>=0.7       P=0.000 R=0.000 F1=0.000 | predicted 0 vs true 78 (bias -78, ratio 0.0)
      agreement: {'extractor_A': 'llm_drafts', 'extractor_B': 'lexicon_detector@0.5', 'both_right': 58, 'only_A_right': 17, 'only_B_right': 32, 'both_wrong': 13}
  child_heard [DEPLOYED]  n=113 gold+=0.301 (model-adjudicated gold)
      llm_drafts                   P=0.676 R=0.676 F1=0.676 | predicted 34 vs true 34 (bias +0, ratio 1.0)
      lexicon_detector@0.5         P=0.377 R=0.588 F1=0.460 | predicted 53 vs true 34 (bias +19, ratio 1.559)
      deployed@conf_cal>=0.7       P=0.676 R=0.676 F1=0.676 | predicted 34 vs true 34 (bias +0, ratio 1.0)
      agreement: {'extractor_A': 'llm_draf

In [1]:
# THRESHOLD SWEEP / COVERAGE CURVE / FALSE-ABSTENTION RATE -----------------------------------
# Chapter 2 obliges the thesis to report BOTH error directions (Wen et al.); the routing metric
# does it for the dispatcher. This is the same measurement for FIELD-LEVEL abstention.
THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
PROBE_THRESHOLD = 0.7          # used only when a field has no deployed default yet


def sweep_field(f):
    """Coverage / precision-of-counted / false-abstention rate across THRESHOLDS, + a figure."""
    fl, fname = f["labels"], f["name"]
    y = fl.gold.to_numpy(int)
    pred = fl.draft_label.astype(int).to_numpy()
    raw = fl.draft_conf.to_numpy(float)
    # calibrated confidence, preferring the honest source: out-of-fold isotonic, so no row is
    # scored by a mapping fitted on its own label. The deployed isotonic is fitted on ALL
    # reviewed rows, so using it here would flatter every threshold.
    conf = oof_isotonic(raw, y, n_splits=5, random_state=42)
    basis = "out-of-fold isotonic (no row scored by a fit that saw its label)"
    if conf is None and f["deployed"] is not None:
        dm = f["deployed"].set_index("id")["conf_cal"]
        conf = fl.id.map(lambda i: float(dm.get(i, 0.0))).to_numpy(float)
        basis = "deployed isotonic (fitted on these rows -- optimistic)"
    if conf is None:
        conf, basis = raw, "raw draft confidence (uncalibrated)"
    thr_default = (f["meta"] or {}).get("threshold")
    op = thr_default if thr_default is not None else PROBE_THRESHOLD
    pos = pred == 1
    n_true = int((y == 1).sum())
    rows = []
    for t in THRESHOLDS + ([op] if op not in THRESHOLDS else []):
        counted = pos & (conf >= t)
        abst = pos & (conf < t)
        n_c = int(counted.sum())
        fa = int((abst & (y == 1)).sum())
        rows.append({"threshold": t, "counted": n_c, "abstained": int(abst.sum()),
                     "coverage": pct(n_c / max(1, int(pos.sum()))),
                     "precision_of_counted": pct(int((counted & (y == 1)).sum()) / n_c) if n_c else None,
                     "false_abstentions": fa,
                     "false_abstention_rate": pct(fa / n_true) if n_true else None})
    rows.sort(key=lambda r: r["threshold"])
    sw = pd.DataFrame(rows)

    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    ax.plot(sw.threshold, sw.coverage, "o-", color="#2c7fb8", label="coverage (of predicted +)")
    ax.plot(sw.threshold, sw.precision_of_counted, "s-", color="#33a02c",
            label="precision of counted")
    ax.plot(sw.threshold, sw.false_abstention_rate, "^-", color="#d95f02",
            label="false-abstention rate (of all real +)")
    ax.axvline(op, color="black", ls="--", lw=1)
    ax.annotate(f"{'deployed default' if thr_default is not None else 'probe'} {op}",
                (op, 0.02), rotation=90, fontsize=8, va="bottom", ha="right")
    ax.set_xlabel("calibrated-confidence threshold"); ax.set_ylabel("rate")
    ax.set_ylim(0, 1.02)
    ax.set_title(f"Abstention trade-off -- {fname} (n={len(fl)} labels)")
    ax.legend(fontsize=8, loc="center left")
    fig.tight_layout()
    out = FIGURES / f"field_{fname}_coverage_threshold.png"
    fig.savefig(out, dpi=130); plt.close(fig)

    at_op = next(r for r in rows if r["threshold"] == op)
    return {"n_labels": int(len(fl)), "predicted_positive": int(pos.sum()),
            "true_positives_in_sample": n_true, "confidence_basis": basis,
            "deployed_default_threshold": thr_default,
            "operating_threshold": op,
            "operating_threshold_is_deployed_default": thr_default is not None,
            "coverage_at_operating_threshold": at_op["coverage"],
            "precision_at_operating_threshold": at_op["precision_of_counted"],
            "false_abstention_rate_at_operating_threshold": at_op["false_abstention_rate"],
            "sweep": rows, "figure": str(out)}


if PRIMARY is None:
    reg_pending("threshold_sweep",
                "Abstention threshold sweep -- coverage, precision of counted, false abstentions",
                "What does raising the confidence threshold actually buy and cost -- how much "
                "precision is gained, and how many real cases are discarded to get it?",
                "For each threshold: coverage of predicted-positive cells, precision of the "
                "cells still counted, and the false-abstention rate. Emitted as a table and a "
                "figure with the operating threshold marked.",
                FIELD_PENDING, "gold", corpus_dependent=False, unblocks=_UNBLOCK)
else:
    sweeps = {f["name"]: sweep_field(f) for f in SCORABLE}
    for name, s in sweeps.items():
        print(f"  {name} (n={s['n_labels']}, predicted+={s['predicted_positive']}, "
              f"real+={s['true_positives_in_sample']}, basis: {s['confidence_basis']})")
        print(pd.DataFrame(s["sweep"]).to_string(index=False))
        print(f"    figure -> {s['figure']}")
    S = sweeps[PRIMARY["name"]]
    lo = next(r for r in S["sweep"] if r["threshold"] == 0.5)
    hi = next(r for r in S["sweep"] if r["threshold"] == 0.7)
    base = S["sweep"][0]
    # three genuinely different shapes, and the takeaway must not describe one it did not get
    moves_at = next((r for r in S["sweep"] if r["coverage"] != base["coverage"]), None)
    inert = moves_at is None
    flat_mid = (not inert and lo["coverage"] == hi["coverage"]
                and lo["precision_of_counted"] == hi["precision_of_counted"])
    if inert:
        _trade = (f"no threshold between {S['sweep'][0]['threshold']} and "
                  f"{S['sweep'][-1]['threshold']} changes anything on `{PRIMARY['name']}` -- "
                  f"coverage stays {base['coverage']}, precision stays "
                  f"{base['precision_of_counted']} and the same "
                  f"{base['false_abstentions']} real cases are discarded throughout, because the "
                  f"calibrated scores are bimodal (they sit at the extremes, not spread across "
                  f"the range). The threshold is therefore inert over the whole sweep: what "
                  f"decides abstention here is the calibration map, not the operating point")
    elif flat_mid:
        _trade = (f"between 0.5 and 0.7 nothing moves on `{PRIMARY['name']}` -- coverage stays "
                  f"{lo['coverage']}, precision stays {lo['precision_of_counted']} and no real "
                  f"case is discarded; the trade only appears at {moves_at['threshold']}, where "
                  f"coverage falls to {moves_at['coverage']} and {moves_at['false_abstentions']} "
                  f"real cases are discarded")
    else:
        _trade = (f"raising the threshold from 0.5 to 0.7 moves the precision of what is still "
                  f"counted from {lo['precision_of_counted']} to {hi['precision_of_counted']} "
                  f"and discards {hi['false_abstentions']} real cases, "
                  f"{pct(float(hi['false_abstention_rate'] or 0) * 100, 1)}% of everything "
                  f"present")
    reg("threshold_sweep",
        f"Abstention threshold sweep -- `{PRIMARY['name']}` coverage, precision, false abstentions",
        "What does raising the confidence threshold actually buy and cost -- how much precision "
        "is gained on the cells still counted, and how many real cases are thrown away to get it?",
        f"For each threshold in {THRESHOLDS}: coverage (counted / predicted-positive), precision "
        f"of the counted cells against the gold, and the false-abstention rate (real positives "
        f"discarded, over all real positives). Run for all {len(SCORABLE)} scorable fields, one "
        f"figure each in figures/, with the operating threshold marked -- the deployed default "
        f"where a field has one, otherwise {PROBE_THRESHOLD} as a stated probe.",
        {"primary_field": PRIMARY["name"],
         "operating_threshold": S["operating_threshold"],
         "operating_threshold_is_deployed_default": S["operating_threshold_is_deployed_default"],
         "coverage_at_operating_threshold": S["coverage_at_operating_threshold"],
         "precision_at_operating_threshold": S["precision_at_operating_threshold"],
         "false_abstention_rate_at_operating_threshold":
             S["false_abstention_rate_at_operating_threshold"],
         "threshold_is_inert_across_the_sweep": inert,
         "coverage_first_changes_at": (moves_at or {}).get("threshold"),
         "per_field": sweeps},
        _trade, "gold",
        f"Both error directions, priced. At the operating threshold {S['operating_threshold']}"
        + ("" if S["operating_threshold_is_deployed_default"] else " (a probe -- this field has "
           "no deployed default yet)")
        + f", `{PRIMARY['name']}` counts {S['coverage_at_operating_threshold']} of its "
          f"predicted positives at precision {S['precision_at_operating_threshold']}, discarding "
          f"{S['false_abstention_rate_at_operating_threshold']} of every real positive in the "
          f"sample. In plain words: {_trade}. "
        + ("Because the threshold is inert here, that operating point is not a choice the "
           "system is currently making -- the abstention it produces comes from the calibration "
           "map, and tuning the threshold would not change a single counted case."
           if inert else
           "The threshold is a choice about which error the answer carries, and the table says "
           "what each choice costs."),
        corpus_dependent=False,
        headline=("false_abstention_rate_at_operating_threshold",
                  S["false_abstention_rate_at_operating_threshold"]),
        backend="ollama", model=field_model(PRIMARY))


  applicant_is_father (n=120, predicted+=57, real+=78, basis: out-of-fold isotonic (no row scored by a fit that saw its label))
 threshold  counted  abstained  coverage  precision_of_counted  false_abstentions  false_abstention_rate
       0.3       57          0     1.000                 0.789                  0                  0.000
       0.4       57          0     1.000                 0.789                  0                  0.000
       0.5       57          0     1.000                 0.789                  0                  0.000
       0.6       57          0     1.000                 0.789                  0                  0.000
       0.7       57          0     1.000                 0.789                  0                  0.000
       0.8        3         54     0.053                 0.667                 43                  0.551
       0.9        0         57     0.000                   NaN                 45                  0.577
    figure -> ../figures/field_a

In [1]:
# LABEL FLIP RATE ----------------------------------------------------------------------------
# The factory DRAFTS labels with a model and something then adjudicates them. The flip rate is
# the integrity check on that protocol: near zero means the drafts were rubber-stamped and any
# F1 against those labels is circular. WHO adjudicated matters as much as how often they
# disagreed, so the provenance recorded in the sidecar is reported next to the rate -- a gold
# set written by a second model is not a human validation, however high the flip rate.
if PRIMARY is None:
    reg_pending("flip_rate", "Label flip rate (annotation integrity)",
                "Were the gold labels genuinely adjudicated, or did the adjudicator agree with "
                "the model's drafts -- which would make any F1 measured against them circular?",
                "flip_rate = #(gold != draft_label) / #labelled, read from the field's labels "
                "file and its sidecar meta.",
                FIELD_PENDING, "gold", corpus_dependent=False, unblocks=_UNBLOCK)
else:
    flips = {}
    for f in SCORABLE:
        fl = f["labels"]
        k = int((fl.gold != fl.draft_label.astype(int)).sum())
        flips[f["name"]] = {
            "n_labels": int(len(fl)), "flips": k, "flip_rate": pct(k / len(fl)),
            "flip_rate_ci95": ci(k, len(fl)),
            "sidecar_flip_rate": (f["meta"] or {}).get("validation", {}).get("flip_rate"),
            "adjudicator": field_adjudicator(f), "labels_provenance": field_provenance(f),
            "drafting_model": field_model(f), "is_deployed": f["is_deployed"],
            "rubber_stamp_warning": bool(k / len(fl) <= 0.02)}
        v = flips[f["name"]]
        print(f"  {f['name']:24s} {v['flips']:3d}/{v['n_labels']:3d} = {v['flip_rate']} "
              f"CI {v['flip_rate_ci95']} | {v['adjudicator']}-adjudicated"
              + ("  !! RUBBER-STAMP WARNING" if v["rubber_stamp_warning"] else ""))
    F = flips[PRIMARY["name"]]
    reg("flip_rate", f"Label flip rate -- `{PRIMARY['name']}` (annotation integrity)",
        "Were the gold labels genuinely adjudicated, or did the adjudicator agree with the "
        "model's drafts -- which would make any F1 measured against them circular?",
        f"flip_rate = #(gold != draft_label) / #labelled, for all {len(SCORABLE)} scorable "
        f"fields, cross-checked against the rate recorded in each sidecar meta. Reported "
        f"together with WHO adjudicated, taken from the sidecar's labels provenance rather than "
        f"assumed. The reference point is the alienation gold set, where 37/120 labels overruled "
        f"the extractor.",
        {"primary_field": PRIMARY["name"], **F, "per_field": flips},
        f"{F['flips']} of {F['n_labels']} drafted labels for `{PRIMARY['name']}` were overruled",
        "gold",
        (f"FLIP RATE {F['flip_rate']} ({F['flips']}/{F['n_labels']}) is at or near zero: the "
         f"adjudicator agreed with essentially every draft, so the content-field F1 measures "
         f"agreement with the drafting model's own output."
         if F["rubber_stamp_warning"] else
         f"Flip rate {F['flip_rate']} ({F['flips']}/{F['n_labels']}, CI {F['flip_rate_ci95']}) on "
         f"`{PRIMARY['name']}`: the drafts were overruled often enough that the labels are "
         f"adjudication rather than a rubber stamp.")
        + (f" But the adjudicator is a MODEL, not a person -- {F['labels_provenance']} -- so a "
           f"high flip rate shows two models disagreeing, not a human validating one. The "
           f"drafts were written by {F['drafting_model']}. No inter-annotator agreement and no "
           f"human gold exists for this field, and no claim of human validation may be made "
           f"from it."
           if F["adjudicator"] == "model" else
           f" Labels provenance: {F['labels_provenance']}.")
        + f" Per-field rates are in the result dict.",
        corpus_dependent=False, headline=("flip_rate", F["flip_rate"]),
        backend="ollama", model=field_model(PRIMARY))


  applicant_is_father       45/120 = 0.375 CI [0.294, 0.464] | model-adjudicated
  child_heard               22/113 = 0.195 CI [0.132, 0.277] | model-adjudicated
  coercive_measures         25/120 = 0.208 CI [0.145, 0.289] | model-adjudicated
  expert_opinion_ordered    29/120 = 0.242 CI [0.174, 0.326] | model-adjudicated
9. LABEL FLIP RATE -- `APPLICANT_IS_FATHER` (ANNOTATION INTEGRITY)
  QUESTION IT ANSWERS : Were the gold labels genuinely adjudicated, or did the adjudicator agree with the model's drafts -- which would make any F1 measured against them circular?
  HOW MEASURED        : flip_rate = #(gold != draft_label) / #labelled, for all 4 scorable fields, cross-checked against the rate recorded in each sidecar meta. Reported together with WHO adjudicated, taken from the sidecar's labels provenance rather than assumed. The reference point is the alienation gold set, where 37/120 labels overruled the extractor.
  CORPUS-DEPENDENT    : False | BACKEND: ollama / llama3.2
  RESULT:
  

## Tier 2 — Label-free intrinsic checks
No annotation at all — properties checkable directly from the data.


In [1]:
# EVIDENCE GROUNDING (ANTI-FABRICATION) ------------------------------------------------------
extf = pd.read_parquet(DATA / "echr_extracted.parquet")
raw_json = (_echr_raw if "_echr_raw" in globals()
            else json.loads((DATA / "echr_parental_alienation.json").read_text()))
text_by_id = {r.get("itemid"): (r.get("full_text") or "") for r in raw_json}
norm = lambda s: re.sub(r"\s+", " ", (s or "")).strip().lower()
checked = grounded = 0
for row in extf.itertuples():
    ev = getattr(row, "alienation_evidence", "") or ""
    if not isinstance(ev, str) or len(ev.strip()) < 20:
        continue
    body = norm(text_by_id.get(row.id, ""))
    for snip in re.split(r"\s*\|\s*|\s*;;\s*|\s*\.\.\.\s*", ev):
        snip = snip.strip().strip('"“”')
        if len(snip) < 20:
            continue
        checked += 1
        grounded += int(norm(snip)[:120] in body)
g_rate = grounded / checked if checked else None
reg("grounding", "Evidence grounding (anti-fabrication)",
    "Are the evidence snippets the extractor cites actually present in the source judgment, or "
    "invented?",
    "For every extracted evidence snippet, verify it appears verbatim (whitespace-normalised, "
    "first 120 characters) in that case's own full text. No LLM, no labels -- a direct "
    "fabrication check, recomputed over the current corpus. The snippets come from "
    "`alienation_evidence`, the only evidence column the extraction table carries today and a "
    "column that is being retired; the CHECK is field-agnostic and will run unchanged on a "
    "factory-deployed field's evidence, but the number below describes that column.",
    {"evidence_column": "alienation_evidence (retired rules extractor)",
     "cases_in_table": int(len(extf)), "snippets_checked": checked,
     "found_verbatim_in_source": grounded, "grounding_rate": pct(g_rate),
     "grounding_rate_ci95": ci(grounded, checked) if checked else None,
     "fabricated": checked - grounded},
    f"{grounded}/{checked} cited snippets found verbatim in their own source case",
    "label-free",
    (f"{grounded}/{checked} snippets verify verbatim ({pct(g_rate)}, CI {ci(grounded, checked)}); "
     f"{checked - grounded} do not. Snippet-level grounding is a necessary condition for a "
     f"faithful extraction, not a sufficient one: a real sentence can still be the wrong "
     f"evidence for the claim it is attached to."
     if checked else
     "No evidence snippets were long enough to check, so this is not a result about grounding "
     "-- it is a report that the check had nothing to run on."),
    corpus_dependent=True, headline=("grounding_rate", pct(g_rate)))


10. EVIDENCE GROUNDING (ANTI-FABRICATION)
  QUESTION IT ANSWERS : Are the evidence snippets the extractor cites actually present in the source judgment, or invented?
  HOW MEASURED        : For every extracted evidence snippet, verify it appears verbatim (whitespace-normalised, first 120 characters) in that case's own full text. No LLM, no labels -- a direct fabrication check, recomputed over the current corpus. The snippets come from `alienation_evidence`, the only evidence column the extraction table carries today and a column that is being retired; the CHECK is field-agnostic and will run unchanged on a factory-deployed field's evidence, but the number below describes that column.
  CORPUS-DEPENDENT    : True
  RESULT:
      - evidence_column: alienation_evidence (retired rules extractor)
      - cases_in_table: 1574
      - snippets_checked: 150
      - found_verbatim_in_source: 150
      - grounding_rate: 1.0
      - grounding_rate_ci95: [0.975, 1.0]
      - fabricated: 0
  EXAMPL

In [1]:
# SHARED PROVENANCE LOAD ----------------------------------------------------------------------
# Every answer-level metric below reads the SAME recorded run: reports/citation_check.json and
# the provenance record each of its rows points at. Loaded once, here, so the funnel, the
# citation checks and the abstention distribution cannot silently describe different answer sets.
# Nothing is re-generated: these are hosted answers that already cost their tokens.
CC = REPORTS / "citation_check.json"
PROV_DIR = REPORTS / "retrieval_provenance"
CC_DATA = json.loads(CC.read_text()) if CC.exists() else None
PROV = []
if CC_DATA:
    for _r in CC_DATA["rows"]:
        _p = Path(_r.get("provenance", ""))
        if not _p.exists():
            continue
        _rec = json.loads(_p.read_text())
        _a = _rec.get("answer") or ""
        _un = None
        if _a.startswith("[generation unavailable"):
            _un = "HTTP 429 (provider token budget)" if "HTTP 429" in _a else _a[:60]
        PROV.append({"cc": _r, "rec": _rec, "mode": _r["packing"],
                     "generated": _un is None and bool(_a.strip()), "unavailable": _un})
PROV_READY = bool(PROV)
# every grounded answer the pipeline has EVER recorded -- a wider, less controlled set than the
# frozen 24, and the only place the abstention layers have actually fired
PROV_ALL = []
for _p in sorted(PROV_DIR.glob("*.json")):
    try:
        PROV_ALL.append(json.loads(_p.read_text()))
    except Exception:
        pass
PACKING_MODES = sorted({d["mode"] for d in PROV})
GEN_BACKENDS = sorted({(d["rec"].get("params") or {}).get("gen_backend") for d in PROV} - {None})
GEN_MODELS = sorted({(d["rec"].get("params") or {}).get("gen_model") for d in PROV} - {None})
PROV_BACKEND = "/".join(GEN_BACKENDS) or None
PROV_MODEL = "/".join(GEN_MODELS) or None
PROV_MISSING = ("reports/citation_check.json is absent -- run "
                "`../.venv/bin/python citation_check.py --limit 24 --modes hybrid,extract`.")
print(f"provenance: {len(PROV)} recorded answers from {CC.name} "
      f"(run {CC_DATA.get('run_at') if CC_DATA else 'n/a'}, modes {PACKING_MODES}) | "
      f"{len(PROV_ALL)} records in {PROV_DIR.name}/ | backend {PROV_BACKEND} / {PROV_MODEL}")


def _g(d, key, default=0):
    return (d["rec"].get("grounding") or {}).get(key, default)


provenance: 48 recorded answers from citation_check.json (run 2026-09-11T19:47:09, modes ['extract', 'hybrid']) | 89 records in retrieval_provenance/ | backend remote / openai/gpt-oss-120b


In [1]:
# EVIDENCE FUNNEL -- retrieved / in prompt / cited ---------------------------------------------
# The three counts an answer reports. They are NOT interchangeable and the first one is the one
# missing everywhere else in this notebook: "how many cases is this grounded in" has one honest
# answer -- cases_cited -- and it is far smaller than the number retrieved. Retrieval and prompt
# construction happen before generation, so their stages are averaged over ALL recorded answers;
# the cited stage is averaged over the answers that actually generated, because an answer that
# never ran cites nothing and would drag the mean down for a reason that is not about grounding.
if not PROV_READY:
    reg_pending("evidence_funnel", "Evidence funnel -- retrieved / in prompt / cited",
                "How much of what retrieval finds actually reaches the generator, and how much "
                "of that is actually cited in the answer?",
                "Mean cases and chunks at each of the three stages, per packing mode, from the "
                "recorded provenance of each answer.",
                PROV_MISSING, "label-free", corpus_dependent=True,
                unblocks="run citation_check.py")
else:
    funnel = {}
    for mode in PACKING_MODES:
        rs = [d for d in PROV if d["mode"] == mode]
        gen = [d for d in rs if d["generated"]]
        ret_cases = [_g(d, "cases_retrieved") for d in rs]
        mean = lambda xs: pct(sum(xs) / len(xs), 2) if xs else None
        funnel[mode] = {
            "answers": len(rs), "answers_generated": len(gen),
            "mean_cases_retrieved": mean(ret_cases),
            "median_cases_retrieved": pct(float(np.median(ret_cases)), 1) if ret_cases else None,
            "min_cases_retrieved": min(ret_cases) if ret_cases else None,
            "max_cases_retrieved": max(ret_cases) if ret_cases else None,
            "mean_cases_in_prompt": mean([_g(d, "cases_in_prompt") for d in rs]),
            "mean_cases_cited": mean([_g(d, "cases_cited") for d in gen]),
            "mean_chunks_retrieved": mean([_g(d, "chunks_retrieved") for d in rs]),
            "mean_chunks_in_prompt": mean([_g(d, "chunks_in_prompt") for d in rs]),
            "mean_chunks_cited": mean([_g(d, "chunks_cited") for d in gen]),
            "cited_stage_basis": f"{len(gen)}/{len(rs)} answers that actually generated",
        }
        f = funnel[mode]
        if f["mean_cases_retrieved"]:
            f["share_of_retrieved_reaching_the_prompt"] = pct(
                f["mean_cases_in_prompt"] / f["mean_cases_retrieved"])
            if f["mean_cases_cited"] is not None:
                f["share_of_retrieved_that_is_cited"] = pct(
                    f["mean_cases_cited"] / f["mean_cases_retrieved"])
                f["share_of_in_prompt_that_is_cited"] = pct(
                    f["mean_cases_cited"] / f["mean_cases_in_prompt"])
        print(f"  {mode:8s} cases  retrieved {f['mean_cases_retrieved']:>6} -> in prompt "
              f"{f['mean_cases_in_prompt']:>6} -> cited {f['mean_cases_cited']} "
              f"({f['cited_stage_basis']})")
        print(f"           chunks retrieved {f['mean_chunks_retrieved']:>6} -> in prompt "
              f"{f['mean_chunks_in_prompt']:>6} -> cited {f['mean_chunks_cited']}")

    _m = max(funnel, key=lambda k: funnel[k]["mean_cases_cited"] or 0)
    _f = funnel[_m]
    reg("evidence_funnel", "Evidence funnel -- retrieved / in prompt / cited",
        "How much of what retrieval finds actually reaches the generator, and how much of that "
        "is actually cited in the answer?",
        f"The three counts every answer reports, averaged over the {len(PROV)} recorded answers "
        f"per packing mode. Retrieved and in-prompt are averaged over ALL answers (both stages "
        f"run before generation); cited is averaged over the answers that actually generated, "
        f"since an answer the provider never returned cites nothing for a reason that has "
        f"nothing to do with grounding. Read from stored provenance -- no answer is re-generated.",
        {"source_file": str(CC), "run_at": CC_DATA.get("run_at"),
         "headline_mode": _m,
         "mean_cases_retrieved": _f["mean_cases_retrieved"],
         "per_packing_mode": funnel},
        f"under `{_m}` packing: {_f['mean_cases_retrieved']} cases retrieved, "
        f"{_f['mean_cases_in_prompt']} put in the prompt, {_f['mean_cases_cited']} cited",
        "label-free",
        f"The funnel is steep and the three counts must not be conflated: under `{_m}` packing "
        f"an answer retrieves {_f['mean_cases_retrieved']} cases on average, "
        f"{_f['mean_cases_in_prompt']} of them reach the prompt "
        f"({_f.get('share_of_retrieved_reaching_the_prompt')} of what was retrieved) and "
        f"{_f['mean_cases_cited']} are cited "
        f"({_f.get('share_of_retrieved_that_is_cited')} of retrieved, "
        f"{_f.get('share_of_in_prompt_that_is_cited')} of what was in front of the model). "
        f"'How many cases is this grounded in' therefore has one honest answer -- the cited "
        f"count -- and quoting the retrieved count in its place overstates the evidence base by "
        f"roughly an order of magnitude.",
        corpus_dependent=True,
        headline=("mean_cases_retrieved", _f["mean_cases_retrieved"]),
        backend=PROV_BACKEND, model=PROV_MODEL)


  extract  cases  retrieved   42.5 -> in prompt  23.88 -> cited 8.25 (24/24 answers that actually generated)
           chunks retrieved  44.75 -> in prompt  24.29 -> cited 8.29
  hybrid   cases  retrieved   42.5 -> in prompt  20.67 -> cited 7.45 (11/24 answers that actually generated)
           chunks retrieved  44.75 -> in prompt   21.0 -> cited 7.45
11. EVIDENCE FUNNEL -- RETRIEVED / IN PROMPT / CITED
  QUESTION IT ANSWERS : How much of what retrieval finds actually reaches the generator, and how much of that is actually cited in the answer?
  HOW MEASURED        : The three counts every answer reports, averaged over the 48 recorded answers per packing mode. Retrieved and in-prompt are averaged over ALL answers (both stages run before generation); cited is averaged over the answers that actually generated, since an answer the provider never returned cites nothing for a reason that has nothing to do with grounding. Read from stored provenance -- no answer is re-generated.
  CORPUS-D

In [1]:
# CITATION AND QUOTE PRECISION --------------------------------------------------------------
# Read from reports/citation_check.json -- the script is NOT re-run here (48 hosted answers).
MIN_QUOTES_TO_REPORT_A_RATE = 20
if not PROV_READY:
    reg_pending("citation_precision", "Citation and quote precision",
                "Do the grounded answers cite ids that exist and that the generator was actually "
                "shown, and are the spans they put in quotation marks really in those sources?",
                "Read from reports/citation_check.json: citation_precision = resolved / "
                "(resolved + invented + real_but_unseen); quote_precision = (checked - "
                "unverified) / checked, per packing mode.",
                PROV_MISSING, "label-free", corpus_dependent=True,
                unblocks="run citation_check.py")
else:
    cc = CC_DATA
    rows_cc = [d["cc"] for d in PROV]
    # WHICH backend produced these answers, and which never generated at all -- both read from
    # the shared provenance load above rather than assumed from the current config.
    unavailable = {}
    for d in PROV:
        if d["unavailable"]:
            unavailable.setdefault(d["mode"], {}).setdefault(d["unavailable"], 0)
            unavailable[d["mode"]][d["unavailable"]] += 1

    per_mode = {}
    for mode in PACKING_MODES:
        rs = [r for r in rows_cc if r["packing"] == mode]
        res = sum(r["citations_ok"] for r in rs)
        inv = sum(len(r["citations_invented"]) for r in rs)
        uns = sum(len(r["citations_real_but_unseen"]) for r in rs)
        tot = res + inv + uns
        qc = sum(r["quotes_checked"] for r in rs)
        qbad = sum(len(r["quotes_unverified"]) for r in rs)
        gone = sum(unavailable.get(mode, {}).values())
        generated = len(rs) - gone
        per_mode[mode] = {
            "questions": len(rs),
            "answers_actually_generated": generated,
            "generation_unavailable": unavailable.get(mode, {}) or None,
            "answers_with_no_citation": sum(1 for r in rs if not r["citations_ok"]),
            "answers_with_no_citation_despite_generating":
                sum(1 for r in rs if not r["citations_ok"]) - gone,
            "citations_resolved": res, "citations_invented": inv,
            "citations_real_but_unseen": uns,
            "citation_precision": pct(res / tot) if tot else None,
            "citation_precision_ci95": ci(res, tot) if tot else None,
            "quotes_checked": qc, "quotes_unverified": qbad,
            "quote_precision": (pct((qc - qbad) / qc) if qc >= MIN_QUOTES_TO_REPORT_A_RATE
                                else None),
            "quote_precision_note": (None if qc >= MIN_QUOTES_TO_REPORT_A_RATE else
                                     f"only {qc} quoted span(s) observed across {len(rs)} "
                                     f"answers -- below the {MIN_QUOTES_TO_REPORT_A_RATE} needed "
                                     f"to state a rate, so no rate is stated. The generator "
                                     f"paraphrases rather than quotes, which is what makes the "
                                     f"denominator small."),
            "mean_cases_cited": pct(sum(r["cases_cited"] for r in rs) / len(rs), 2),
            "mean_cases_in_prompt": pct(sum(r["cases_in_prompt"] for r in rs) / len(rs), 2),
        }
        m = per_mode[mode]
        print(f"  {mode:8s} citation precision {m['citation_precision']} CI "
              f"{m['citation_precision_ci95']} ({inv} invented, {uns} unseen of {tot}) | "
              f"quotes {qc} checked -> rate {m['quote_precision']} | cases cited "
              f"{m['mean_cases_cited']} of {m['mean_cases_in_prompt']} in prompt")
        if m["generation_unavailable"]:
            print(f"           {gone}/{len(rs)} answers never generated: "
                  f"{m['generation_unavailable']} -- those contribute no citations and are NOT "
                  f"a citation failure")

    _best_mode = max(per_mode, key=lambda k: (per_mode[k]["citation_precision"] or 0))
    _qtotal = sum(m["quotes_checked"] for m in per_mode.values())
    reg("citation_precision", "Citation and quote precision",
        "Do the grounded answers cite ids that exist and that the generator was actually shown, "
        "and are the spans they put in quotation marks really in those sources?",
        "Read (not re-run) from reports/citation_check.json. citation_precision = resolved / "
        "(resolved + invented + real_but_unseen); quote_precision = (checked - unverified) / "
        "checked. Reported per packing mode, with the mean cases cited against the mean cases "
        "actually in the prompt. Answers where generation never ran are identified from each "
        "answer's own provenance record and counted separately, because a missing answer cites "
        "nothing and is not a citation failure. The backend is read from those same provenance "
        "records rather than from the current config.",
        {"source_file": str(CC), "run_at": cc.get("run_at"),
         "best_citation_precision": per_mode[_best_mode]["citation_precision"],
         "best_citation_precision_ci95": per_mode[_best_mode]["citation_precision_ci95"],
         "best_citation_precision_mode": _best_mode,
         "per_packing_mode": per_mode,
         "quote_precision_floor": MIN_QUOTES_TO_REPORT_A_RATE,
         "total_quotes_observed": _qtotal},
        "an id the answer assembled by incrementing a real chunk suffix is a fabricated "
        "reference even when the sentence it supports is true",
        "label-free",
        f"Citation precision is {per_mode[_best_mode]['citation_precision']} "
        f"(CI {per_mode[_best_mode]['citation_precision_ci95']}) under `{_best_mode}` packing, "
        f"so invented and never-shown ids are rare but not zero. Quote precision is NOT reported "
        f"as a rate: {_qtotal} quoted span(s) were observed in total, far below the "
        f"{MIN_QUOTES_TO_REPORT_A_RATE} a rate would need. Answers that never generated are "
        f"separated out per mode, so a provider outage cannot be read as a citation failure.",
        corpus_dependent=True,
        headline=("best_citation_precision", per_mode[_best_mode]["citation_precision"]),
        backend=PROV_BACKEND, model=PROV_MODEL)


  extract  citation precision 0.966 CI [0.932, 0.983] (6 invented, 1 unseen of 206) | quotes 1 checked -> rate None | cases cited 8.25 of 23.88 in prompt
  hybrid   citation precision 0.976 CI [0.917, 0.993] (1 invented, 1 unseen of 84) | quotes 0 checked -> rate None | cases cited 3.42 of 20.67 in prompt
           13/24 answers never generated: {'HTTP 429 (provider token budget)': 13} -- those contribute no citations and are NOT a citation failure
12. CITATION AND QUOTE PRECISION
  QUESTION IT ANSWERS : Do the grounded answers cite ids that exist and that the generator was actually shown, and are the spans they put in quotation marks really in those sources?
  HOW MEASURED        : Read (not re-run) from reports/citation_check.json. citation_precision = resolved / (resolved + invented + real_but_unseen); quote_precision = (checked - unverified) / checked. Reported per packing mode, with the mean cases cited against the mean cases actually in the prompt. Answers where generation never

In [1]:
# ANSWER-LEVEL FAITHFULNESS CHECKS -- jurisdiction attribution + citation aptness --------------
# Two checks the id-realness test cannot see. jurisdiction_check(): a citation can be real, in
# the prompt, and still be attached to the wrong legal system -- in a three-jurisdiction corpus
# that is the worst error available. citation_aptness(): a citation can be real, in the prompt
# and correctly attributed and still not support the line it sits on, which is what catches one
# id-set stapled onto every proposition. Both are computed at answer time and stored; they are
# read here, not recomputed.
APT_UNCOVERED = ["generation never ran", "no resolvable citation",
                 "cited by DOCUMENT id only", "cited by POSITION only", "other"]
if not PROV_READY:
    reg_pending("answer_checks",
                "Answer-level faithfulness -- jurisdiction attribution and citation aptness",
                "Is a real, in-prompt citation attached to the right jurisdiction, and does it "
                "actually support the proposition it is attached to?",
                "jurisdiction_check() mismatches and citation_aptness() rank outcomes, read "
                "from each answer's stored provenance record.",
                PROV_MISSING, "label-free", corpus_dependent=True,
                unblocks="run citation_check.py")
else:
    jur, apt = {}, {}
    examples, flagged_ex = [], []
    for mode in PACKING_MODES:
        rs = [d for d in PROV if d["mode"] == mode]
        gen = [d for d in rs if d["generated"]]
        mm = [(d, _g(d, "jurisdiction_mismatches", []) or []) for d in gen]
        n_mm = sum(len(x) for _, x in mm)
        affected = sum(1 for _, x in mm if x)
        cited_total = sum(len(_g(d, "cited_chunk_ids", []) or []) for d in gen)
        jur[mode] = {"answers_generated": len(gen), "answers_with_a_mismatch": affected,
                     "mismatches": n_mm, "citations_resolved": cited_total,
                     "mismatch_rate_per_citation": pct(n_mm / cited_total) if cited_total else None,
                     "mismatch_rate_per_citation_ci95": ci(n_mm, cited_total) if cited_total else None}
        for d, x in mm:
            for row in x[:2]:
                examples.append({"mode": mode, "qid": d["cc"]["qid"],
                                 "claimed": row.get("claimed"), "actual": row.get("actual"),
                                 "chunk_id": row.get("chunk_id")})
        # aptness, and WHY it could not run where it did not
        cov = {k: 0 for k in APT_UNCOVERED}
        checked = ok = 0
        basis = None
        for d in rs:
            ap = d["rec"].get("aptness")
            if ap:
                checked += ap["citations_checked"]; ok += ap["apt"]
                basis = basis or ap.get("basis")
                for fl in (ap.get("flagged") or [])[:1]:
                    flagged_ex.append({"mode": mode, "qid": d["cc"]["qid"],
                                       "rank": fl.get("rank"), "of": fl.get("of"),
                                       "proposition": (fl.get("proposition") or "")[:110]})
                continue
            g = d["rec"].get("grounding") or {}
            if not d["generated"]:
                cov["generation never ran"] += 1
            elif not d["cc"]["citations_ok"]:
                cov["no resolvable citation"] += 1
            elif g.get("citations_by_position"):
                cov["cited by POSITION only"] += 1
            elif g.get("citations_by_document"):
                cov["cited by DOCUMENT id only"] += 1
            else:
                cov["other"] += 1
        covered = len(rs) - sum(cov.values())
        apt[mode] = {"answers": len(rs), "answers_covered": covered,
                     "not_covered": {k: v for k, v in cov.items() if v},
                     "citations_checked": checked, "apt": ok, "not_apt": checked - ok,
                     "apt_rate": pct(ok / checked) if checked else None,
                     "apt_rate_ci95": ci(ok, checked) if checked else None,
                     "basis": basis}
        print(f"  {mode:8s} jurisdiction: {jur[mode]['mismatches']} mismatch(es) across "
              f"{affected}/{len(gen)} generated answers, {cited_total} citations "
              f"(rate {jur[mode]['mismatch_rate_per_citation']})")
        print(f"           aptness: {apt[mode]['apt_rate']} over {checked} citations in "
              f"{covered}/{len(rs)} answers | not covered: {apt[mode]['not_covered']}")
    for e in examples[:4]:
        print(f"    MISATTRIBUTED [{e['mode']} {e['qid']}] claimed {e['claimed']} but "
              f"{e['chunk_id']} is {e['actual']}")

    _tot_mm = sum(v["mismatches"] for v in jur.values())
    _tot_cit = sum(v["citations_resolved"] for v in jur.values())
    _tot_chk = sum(v["citations_checked"] for v in apt.values())
    _tot_apt = sum(v["apt"] for v in apt.values())
    _cov = sum(v["answers_covered"] for v in apt.values())
    reg("answer_checks",
        "Answer-level faithfulness -- jurisdiction attribution and citation aptness",
        "Is a real, in-prompt citation attached to the right jurisdiction, and does it actually "
        "support the proposition it is attached to?",
        f"Both checks run at answer time and are read here from the stored provenance of the "
        f"same {len(PROV)} recorded answers the citation-precision metric uses. "
        f"jurisdiction_check() compares the nearest jurisdiction label before each citation "
        f"against the cited chunk's actual jurisdiction -- a heuristic by construction. "
        f"citation_aptness() ranks the in-prompt chunks by similarity to each proposition and "
        f"flags a cited chunk that falls outside the top ranks. Aptness can only run on "
        f"citations written as CHUNK IDS, so its coverage and the reason for every uncovered "
        f"answer are reported rather than left as a silent denominator.",
        {"source_file": str(CC), "run_at": CC_DATA.get("run_at"),
         "jurisdiction_mismatches_total": _tot_mm,
         "citations_resolved_total": _tot_cit,
         "jurisdiction_mismatch_rate": pct(_tot_mm / _tot_cit) if _tot_cit else None,
         "jurisdiction_mismatch_rate_ci95": ci(_tot_mm, _tot_cit) if _tot_cit else None,
         "jurisdiction_per_packing_mode": jur,
         "jurisdiction_examples": examples[:6],
         "aptness_answers_covered": _cov, "aptness_answers_total": len(PROV),
         "aptness_citations_checked": _tot_chk, "aptness_apt": _tot_apt,
         "aptness_rate": pct(_tot_apt / _tot_chk) if _tot_chk else None,
         "aptness_rate_ci95": ci(_tot_apt, _tot_chk) if _tot_chk else None,
         "aptness_per_packing_mode": apt,
         "aptness_flagged_examples": flagged_ex[:6]},
        (f"claimed CH, cited {examples[0]['chunk_id']} which is {examples[0]['actual']}"
         if examples else "no misattribution observed"),
        "label-free",
        f"Two failures a realness check cannot see, both present. "
        f"{_tot_mm} citation(s) of {_tot_cit} are attached to the wrong jurisdiction "
        f"(rate {pct(_tot_mm / _tot_cit) if _tot_cit else None}, "
        f"CI {ci(_tot_mm, _tot_cit) if _tot_cit else None}) -- in a three-jurisdiction corpus "
        f"that is a misattribution of one legal system's rule to another, not a formatting slip. "
        f"Aptness is {pct(_tot_apt / _tot_chk) if _tot_chk else None} "
        f"(CI {ci(_tot_apt, _tot_chk) if _tot_chk else None}) over {_tot_chk} citations, so "
        f"roughly a third of cited passages do not rank as support for the line they sit on. "
        f"That figure covers only {_cov} of {len(PROV)} answers, and the gap is not random: the "
        f"check reads chunk-id citations, and answers that cited by document id or by position "
        f"are invisible to it. Coverage and its reasons are in the result dict; the aptness rate "
        f"is a property of the answers it could read, not of all of them.",
        corpus_dependent=True,
        headline=("aptness_rate", pct(_tot_apt / _tot_chk) if _tot_chk else None),
        backend=PROV_BACKEND, model=PROV_MODEL)


  extract  jurisdiction: 12 mismatch(es) across 5/24 generated answers, 199 citations (rate 0.06)
           aptness: 0.605 over 167 citations in 15/24 answers | not covered: {'no resolvable citation': 2, 'cited by DOCUMENT id only': 5, 'cited by POSITION only': 2}
  hybrid   jurisdiction: 2 mismatch(es) across 1/11 generated answers, 82 citations (rate 0.024)
           aptness: 0.728 over 125 citations in 10/24 answers | not covered: {'generation never ran': 13, 'cited by DOCUMENT id only': 1}
    MISATTRIBUTED [extract q17] claimed CH but ris:JJR_20060913_OGH0002_0030OB00111_06D0000_002:0 is AT (OGH)
    MISATTRIBUTED [extract q17] claimed CH but ris:JJR_19991123_OGH0002_0040OB00311_99K0000_001:0 is AT (OGH)
    MISATTRIBUTED [extract q18] claimed CH but ris:JJR_19731024_OGH0002_0050OB00185_7300000_001:0 is AT (OGH)
    MISATTRIBUTED [extract q18] claimed CH but ris:JJR_19970410_OGH0002_0060OB02398_96G0000_001:0 is AT (OGH)
13. ANSWER-LEVEL FAITHFULNESS -- JURISDICTION ATTRIBUTION A

In [1]:
# DETERMINISM -------------------------------------------------------------------------------
q_det = "How many cases against Poland are in the corpus?"
r1, r2 = predict_bucket(q_det), predict_bucket(q_det)
_sql = "SELECT respondent, COUNT(*) n FROM echr_meta GROUP BY respondent ORDER BY respondent"
s1, s2 = run_sql(_sql), run_sql(_sql)
h1 = [h["chunk_id"] for h in retrieve("enforcement of contact rights", k=10)]
h2 = [h["chunk_id"] for h in retrieve("enforcement of contact rights", k=10)]
det = bool(r1 == r2 and s1.equals(s2) and h1 == h2)
print(f"router {r1 == r2} | sql {s1.equals(s2)} | retrieval {h1 == h2} -> deterministic {det}")
reg("determinism", "Determinism of the deterministic paths",
    "Do the paths that are supposed to be reproducible -- routing, SQL execution, retrieval "
    "ranking -- return identical results on re-run, so the evaluation itself is stable?",
    "Run the dispatch decision, a metadata query and a retrieval twice each; assert identical "
    "results. This covers the DETERMINISTIC paths only. It says nothing about the LLM paths, "
    "and deliberately makes no claim about them.",
    {"router_stable": bool(r1 == r2), "sql_stable": bool(s1.equals(s2)),
     "retrieval_stable": bool(h1 == h2), "deterministic_paths_stable": det,
     "llm_paths_covered": False,
     "llm_reproducibility": "by cache, not by construction -- local models run at temperature 0 "
                            "but a hosted provider can change the weights behind a model string "
                            "without notice, so a hosted figure is re-derivable only from the "
                            "stored SQL / provenance / citation-check artefacts",
     "generation_backend": f"{GEN_BACKEND}/{GEN_MODEL_STR}",
     "nl2sql_backend": f"ollama/{NL2SQL_MODEL}"},
    "same question -> same route; same query -> same table; same retrieval -> same chunk ids",
    "label-free",
    f"The deterministic paths are stable (router {r1 == r2}, SQL {s1.equals(s2)}, retrieval "
    f"{h1 == h2}), so every non-LLM figure in this notebook re-derives exactly. The LLM-produced "
    f"figures (citation precision, answer-level checks, synthetic-QA accuracy) are reproducible "
    f"from their cached artefacts rather than by re-running the model, "
    f"the model, and are labelled with the backend and model that produced them.",
    corpus_dependent=True, headline=("deterministic_paths_stable", det))


router True | sql True | retrieval True -> deterministic True
14. DETERMINISM OF THE DETERMINISTIC PATHS
  QUESTION IT ANSWERS : Do the paths that are supposed to be reproducible -- routing, SQL execution, retrieval ranking -- return identical results on re-run, so the evaluation itself is stable?
  HOW MEASURED        : Run the dispatch decision, a metadata query and a retrieval twice each; assert identical results. This covers the DETERMINISTIC paths only. It says nothing about the LLM paths, and deliberately makes no claim about them.
  CORPUS-DEPENDENT    : True
  RESULT:
      - router_stable: True
      - sql_stable: True
      - retrieval_stable: True
      - deterministic_paths_stable: True
      - llm_paths_covered: False
      - llm_reproducibility: by cache, not by construction -- local models run at temperature 0 but a hosted provider can change the weights behind a model string without notice, so a hosted figure is re-derivable only from the stored SQL / provenance / citat

## Tier 3 — Programmatic ground truth
Truth generated by the database — scales to any size without labels.


In [1]:
# GOLD-SQL DENOTATION -----------------------------------------------------------------------
if "QUERIES" not in globals():
    _qnb = json.loads(Path("echr_query.ipynb").read_text())
    for _c in _qnb["cells"]:
        if _c["cell_type"] == "code" and "QUERIES = {" in "".join(_c["source"]):
            with contextlib.redirect_stdout(io.StringIO()):
                try:
                    exec("".join(_c["source"]), globals())
                except Exception as e:
                    print("canned-query cell failed to exec:", e)
            break
QUERIES = globals().get("QUERIES", {})
ok, broken = 0, []
for key, spec in QUERIES.items():
    sql = (spec.get("sql") or spec.get("query")) if isinstance(spec, dict) else spec
    try:
        con.execute("EXPLAIN " + sql)
        run_sql(sql)
        ok += 1
    except Exception as e:
        broken.append({"key": key, "error": str(e).splitlines()[0][:160]})
print(f"canned queries: {ok}/{len(QUERIES)} EXPLAIN-validate and execute against the live DB")
for b in broken:
    print("  BROKEN:", b)
reg("gold_sql", "Gold-SQL denotation",
    "Are the reviewed, citable canned queries all valid and executable against the LIVE database "
    "-- the trusted reference the generated NL->SQL is compared against?",
    "For each canned query: EXPLAIN-validate against the current schema, then execute; count how "
    "many pass. No LLM. Re-run against the re-imported corpus, so a query that silently broke on "
    "a schema change shows up here.",
    {"canned_queries": len(QUERIES), "valid_and_execute": ok,
     "broken": broken or None,
     "echr_cases_in_db": N_ECHR},
    "e.g. 'cases per respondent state' and 'violation rate by state' both validate and run",
    "programmatic",
    f"{ok} of {len(QUERIES)} reviewed canned queries validate and execute against the current "
    f"{N_ECHR}-case database"
    + ("" if not broken else f"; {len(broken)} do not and are listed in the result dict")
    + ". The reference set the NL->SQL path is scored against is therefore sound (or its "
      "breakages are named), which is what makes the synthetic-QA metric meaningful.",
    corpus_dependent=True, headline=("valid_and_execute", ok))


canned queries: 9/9 EXPLAIN-validate and execute against the live DB
15. GOLD-SQL DENOTATION
  QUESTION IT ANSWERS : Are the reviewed, citable canned queries all valid and executable against the LIVE database -- the trusted reference the generated NL->SQL is compared against?
  HOW MEASURED        : For each canned query: EXPLAIN-validate against the current schema, then execute; count how many pass. No LLM. Re-run against the re-imported corpus, so a query that silently broke on a schema change shows up here.
  CORPUS-DEPENDENT    : True
  RESULT:
      - canned_queries: 9
      - valid_and_execute: 9
      - broken: None
      - echr_cases_in_db: 1574
  EXAMPLE : e.g. 'cases per respondent state' and 'violation rate by state' both validate and run
  TAKEAWAY: 9 of 9 reviewed canned queries validate and execute against the current 1574-case database. The reference set the NL->SQL path is scored against is therefore sound (or its breakages are named), which is what makes the synthetic-

In [1]:
# SYNTHETIC-QA EXECUTION ACCURACY -----------------------------------------------------------
# The only metric here immune to the corpus re-import: the gold answers are recomputed FROM the
# live database every run, so growing the corpus moves the questions and the truth together.
#
# WHY THIS STAYS ON LOCAL OLLAMA. Two reasons, and the second is the load-bearing one.
#  (a) There is no speed gain. The schema + few-shot prefix (~1,959 tok) is constant and only the
#      question changes at the end, so llama.cpp reuses the KV cache: ~113 s cold, ~15 s warm. 30
#      questions therefore cost ~9 min locally. On the hosted free tier the same 30 calls are
#      bounded by an 8,000 tokens-per-minute budget against a ~2,000-token prompt -- about 4 calls
#      a minute, ~8 min. The two are the same.
#  (b) The local model IS the finding. echr_query.ipynb records measured behaviour of small local
#      translators: a general 3B copies few-shot answers and drops filters; a coder 3B translates
#      entities, tables and composed conditions but still invents plausible filters on harder
#      questions; and offering a CANNOT_ANSWER escape makes it refuse anything needing
#      composition (the lazy-escape effect). Those results are the justification for the
#      deterministic nets -- deny-list, required filters, EXPLAIN validation, GROUP BY check --
#      and for the rule that refusal is never delegated to the model. Running this on a 120B
#      hosted reasoner would stop the nets firing and strand the evidence for the design.
SQL_CACHE = DATA / "synthetic_qa_cache.json"
_ISO3_NAME = dict(re.findall(r"\b([A-Z]{3})=([A-Z][a-z]+)", SCHEMA))   # from the deployed SCHEMA


def _denote(df):
    """Denotation of a result set: order-insensitive multiset of rounded row tuples.

    Execution accuracy compares what the query RETURNS, not how it is written, so two different
    statements that denote the same table both count as correct. Column ORDER still matters --
    a row (canton, n) is not the row (n, canton) -- which is the conventional strictness.
    """
    if df is None or len(df) == 0:
        return []
    out = []
    for row in df.itertuples(index=False, name=None):
        out.append(tuple(round(float(v), 3) if isinstance(v, (int, float, np.floating,
                                                              np.integer)) and not isinstance(v, bool)
                         else str(v) for v in row))
    return sorted(out)


def synth_pairs():
    """(question, gold_sql, gold_denotation) triples the database already knows the answer to.

    Deliberately broader than value substitution: a year filter, two-condition questions, grouped
    breakdowns and questions over the German-corpus tables, so the harness exercises TABLE
    SELECTION and composition rather than only entity mapping.
    """
    out = []

    def add(q, sql):
        try:
            out.append((q, sql, _denote(con.execute(sql).df())))
        except Exception as e:
            print("  generator skipped:", q, "->", str(e).splitlines()[0][:90])

    # A. value substitution inside one table (the base case)
    for (c,) in con.execute("SELECT canton FROM swiss_meta WHERE canton IS NOT NULL "
                            "GROUP BY canton ORDER BY COUNT(*) DESC LIMIT 8").fetchall():
        add(f"How many Swiss decisions are from canton {c}?",
            f"SELECT COUNT(*) AS n FROM swiss_meta WHERE canton = '{c}'")
    # B. country NAME -> ISO-3 respondent code, and the respondent reading of a country
    _resp = [s for (s,) in con.execute(
        "SELECT respondent FROM echr_meta WHERE respondent IS NOT NULL GROUP BY respondent "
        "ORDER BY COUNT(*) DESC LIMIT 14").fetchall() if s in _ISO3_NAME][:8]
    for s in _resp:
        add(f"How many ECHR cases are against {_ISO3_NAME[s]}?",
            f"SELECT COUNT(*) AS n FROM echr WHERE respondent_state = '{s}'")
    # C. year filter
    for (yr,) in con.execute("SELECT year FROM echr WHERE year IS NOT NULL GROUP BY year "
                             "ORDER BY COUNT(*) DESC LIMIT 5").fetchall():
        add(f"How many ECHR cases were decided in {int(yr)}?",
            f"SELECT COUNT(*) AS n FROM echr WHERE year = {int(yr)}")
    # D. two composed conditions
    for s in _resp[:4]:
        add(f"How many merits judgments against {_ISO3_NAME[s]} found a violation?",
            f"SELECT COUNT(*) AS n FROM echr WHERE genre = 'merits' AND outcome = 'violation' "
            f"AND respondent_state = '{s}'")
    # E. grouped breakdowns (an ungrouped total here is a silent substitution, not a near miss)
    add("How many ECHR cases are there per outcome?",
        "SELECT outcome, COUNT(*) AS n FROM echr GROUP BY outcome")
    add("How many ECHR cases are there per genre?",
        "SELECT genre, COUNT(*) AS n FROM echr GROUP BY genre")
    add("How many Swiss decisions are there per court type?",
        "SELECT court_type, COUNT(*) AS n FROM swiss_meta GROUP BY court_type")
    # F. the German corpora -- TABLE selection, not value substitution
    add("How many Austrian OGH records are of dokumenttyp Rechtssatz?",
        "SELECT COUNT(*) AS n FROM ris_meta WHERE dokumenttyp = 'Rechtssatz'")
    add("How many Austrian OGH records are there per dokumenttyp?",
        "SELECT dokumenttyp, COUNT(*) AS n FROM ris_meta GROUP BY dokumenttyp")
    for (yr,) in con.execute("SELECT year FROM swiss_meta WHERE year IS NOT NULL GROUP BY year "
                             "ORDER BY COUNT(*) DESC LIMIT 2").fetchall():
        add(f"How many Swiss decisions are from {int(yr)}?",
            f"SELECT COUNT(*) AS n FROM swiss_meta WHERE year = {int(yr)}")
    _c2 = con.execute("SELECT canton FROM swiss_meta WHERE canton IS NOT NULL AND year IS NOT NULL "
                      "GROUP BY canton ORDER BY COUNT(*) DESC LIMIT 2").fetchall()
    for (c,) in _c2:
        add(f"Wie viele Schweizer Entscheidungen aus dem Kanton {c} stammen aus dem Jahr 2023?",
            f"SELECT COUNT(*) AS n FROM swiss_meta WHERE canton = '{c}' AND year = 2023")
    return out


SYNTH = synth_pairs()
print(f"generated {len(SYNTH)} self-checking (question, gold-SQL, gold-answer) triples")


generated 34 self-checking (question, gold-SQL, gold-answer) triples


In [1]:
# (continued) -- run or score from cache, then classify the three outcomes -------------------
def _ollama_reachable():
    """Re-tested HERE, not taken from boot: `ollama serve` may have been started since."""
    try:
        import ollama
        ollama.list()
        return True
    except Exception:
        return False


OLLAMA_NOW = _ollama_reachable()
cache = json.loads(SQL_CACHE.read_text()) if SQL_CACHE.exists() else {}
cached_hits = sum(1 for q, _, _ in SYNTH if q in cache)
print(f"ollama reachable: {OLLAMA_NOW} | cached SQL for {cached_hits}/{len(SYNTH)} questions")

live, warm_s = None, None
if OLLAMA_NOW or cached_hits:
    if OLLAMA_NOW:
        # warm the constant prefix first, so the ~113 s cold cost is not charged to question 1
        _t = time.time()
        with contextlib.redirect_stdout(io.StringIO()):
            nl2sql("How many cases are in the corpus?")
        warm_s = round(time.time() - _t, 1)
        print(f"prefix warmed in {warm_s}s (this call is discarded, not scored)")

    # scoring from cache scores only what the cache holds: an uncached question is a MISSING
    # measurement, not a refusal, and counting it as one would flatter the refusal breakdown
    SCORED = SYNTH if OLLAMA_NOW else [t for t in SYNTH if t[0] in cache]
    not_scored = len(SYNTH) - len(SCORED)
    if not_scored:
        print(f"scoring {len(SCORED)}/{len(SYNTH)} questions from cache; {not_scored} have no "
              f"cached SQL and are excluded from the denominator, not counted as refusals")
    outcomes, per_q_rows, times = {"correct": 0, "wrong_number": 0, "refused_or_untranslated": 0}, [], []
    refusal_kinds = {}
    for q, gold_sql, gold_ans in SCORED:
        entry = cache.get(q)
        if OLLAMA_NOW:
            hint, must = _sql_route(q)          # the deployed Bucket-2 path, hint and all
            t0 = time.time()
            with contextlib.redirect_stdout(io.StringIO()):
                sql = nl2sql(q, hint=hint, must_filter=must)
            times.append(time.time() - t0)
            entry = {"sql": sql, "backend": "ollama", "model": NL2SQL_MODEL,
                     "generated_at": datetime.now().isoformat(timespec="seconds"),
                     "hint_used": bool(hint), "must_filter": [list(m) for m in (must or [])],
                     "group_by_required": bool(requested_group_by(q))}
            cache[q] = entry
        sql = (entry or {}).get("sql")
        if not sql or str(sql).startswith("--"):
            outcomes["refused_or_untranslated"] += 1
            kind = (str(sql).split(":")[0].strip("- ") if sql else "no SQL produced")
            refusal_kinds[kind] = refusal_kinds.get(kind, 0) + 1
            verdict = "refused_or_untranslated"
        else:
            try:
                got = _denote(con.execute(sql).df())
                verdict = "correct" if got == gold_ans else "wrong_number"
            except Exception as e:
                verdict = "refused_or_untranslated"
                kind = "execution failed: " + str(e).splitlines()[0][:60]
                refusal_kinds[kind] = refusal_kinds.get(kind, 0) + 1
            outcomes[verdict] += 1
        per_q_rows.append({"question": q, "verdict": verdict, "sql": sql, "gold_sql": gold_sql})
        print(f"  {verdict:24s} {q[:66]}")

    if OLLAMA_NOW:
        SQL_CACHE.write_text(json.dumps(cache, ensure_ascii=False, indent=1))
        print(f"cached {len(cache)} generated statements -> {SQL_CACHE}")

    # which deterministic nets were actually exercised -- an accuracy figure over questions that
    # triggered none of them says nothing about the nets
    nets = {"source_route_hint": sum(1 for q, _, _ in SCORED if cache.get(q, {}).get("hint_used")),
            "required_filter": sum(1 for q, _, _ in SCORED if cache.get(q, {}).get("must_filter")),
            "group_by_required": sum(1 for q, _, _ in SCORED
                                     if cache.get(q, {}).get("group_by_required"))}
    nq = len(SCORED)
    k = outcomes["correct"]
    translated = k + outcomes["wrong_number"]
    live = {"n_questions": nq,
            "answered_correctly": k,
            "answered_with_a_wrong_number": outcomes["wrong_number"],
            "refused_or_untranslated": outcomes["refused_or_untranslated"],
            "refusal_breakdown": refusal_kinds or None,
            "questions_triggering_each_net": nets,
            "execution_accuracy": pct(k / nq),
            "execution_accuracy_ci95": ci(k, nq),
            "accuracy_among_translated": pct(k / translated) if translated else None,
            "generated_but_not_scored": not_scored,
            "scored_from": "live ollama run" if OLLAMA_NOW else f"cache ({SQL_CACHE.name})",
            "prefix_warm_seconds": warm_s,
            "mean_seconds_per_question": pct(float(np.mean(times)), 1) if times else None,
            "sql_cache": str(SQL_CACHE)}
    print(f"\nexecution accuracy {k}/{nq} = {live['execution_accuracy']} CI "
          f"{live['execution_accuracy_ci95']} | wrong number "
          f"{outcomes['wrong_number']} | refused/untranslated "
          f"{outcomes['refused_or_untranslated']}")

_how = ("Auto-generate (question, gold-SQL, gold-answer) triples from the live DB -- value "
        "substitution, a year filter, two-condition questions, grouped breakdowns and questions "
        "over the German-corpus tables -- then run the DEPLOYED Bucket-2 path (_sql_route + "
        "nl2sql with hint and required filters) and compare the DENOTATION of the generated "
        "statement against the SQL-computed truth. The three outcomes are kept apart: a "
        "translator that declines is not making the same mistake as one that returns a plausible "
        "wrong count. Runs on local ollama (see the cell comment for why it stays there); the "
        "generated SQL is cached per question so a re-run scores the same statements and the "
        "query behind the number stays inspectable.")
if live is None:
    reg_pending("synth_harness", "Synthetic-QA execution accuracy (label-free, at scale)",
                "Can the NL->SQL layer be tested at scale with zero human labels -- by generating "
                "questions whose true answer the database already knows?",
                _how,
                f"The harness IS implemented and generated {len(SYNTH)} self-checking "
                f"(question, gold-SQL, gold-answer) triples from the live database; only the "
                f"scoring leg is unrun. ollama is unreachable and {SQL_CACHE.name} holds no "
                f"cached SQL for these questions, so neither a live run nor a cached scoring is "
                f"possible. Start `ollama serve` (model {NL2SQL_MODEL}) and re-run this cell. "
                f"No accuracy figure may be quoted for the NL->SQL layer until then.",
                "programmatic", corpus_dependent=True,
                unblocks="start ollama serve")
else:
    reg("synth_harness", "Synthetic-QA execution accuracy (label-free, at scale)",
        "Can the NL->SQL layer be tested at scale with zero human labels -- by generating "
        "questions whose true answer the database already knows?",
        _how,
        {"auto_generated_qa_pairs": len(SYNTH), **live},
        f"'{SYNTH[0][0]}' -> the DB already knows the answer, so correctness is automatic",
        "programmatic",
        (f"Execution accuracy {live['execution_accuracy']} "
         f"(CI {live['execution_accuracy_ci95']}, n={live['n_questions']}) with "
         f"{live['answered_with_a_wrong_number']} wrong numbers and "
         f"{live['refused_or_untranslated']} refusals -- a CEILING, which means the harness at "
         f"this difficulty no longer discriminates: the limit is the generator, not the "
         f"translator. What it does establish is that entity translation, ISO-3 mapping, year "
         f"filters, two-condition composition, grouped breakdowns and table selection across "
         f"three corpora are all handled correctly by a 3B coder model behind the deployed nets "
         f"(a source-route hint fired on {nets['source_route_hint']} questions, a required "
         f"filter on {nets['required_filter']}, a GROUP BY requirement on "
         f"{nets['group_by_required']}). The known failure modes -- invented filters, the "
         f"lazy-escape refusal -- are recorded in echr_query.ipynb on harder compositions and "
         f"are not reproduced at this difficulty."
         if live['answered_with_a_wrong_number'] == 0 and live['refused_or_untranslated'] == 0 else
         f"Execution accuracy {live['execution_accuracy']} "
         f"(CI {live['execution_accuracy_ci95']}, n={live['n_questions']}), and the errors split "
         f"{live['answered_with_a_wrong_number']} plausible-but-wrong numbers against "
         f"{live['refused_or_untranslated']} refusals or failed translations. That split is the "
         f"result, not the average: the deterministic nets are built to convert the first kind "
         f"into the second, and only a wrong number reaches a reader as an answer.")
        + f" The generated SQL is cached at {SQL_CACHE.name}, so the query behind every verdict "
          f"is inspectable -- for an aggregate answer the trust boundary is the query text, not "
          f"the number."
        + (f" Timing corroborates the backend decision: the constant prefix cost "
           f"{live['prefix_warm_seconds']}s once and every scored question "
           f"{live['mean_seconds_per_question']}s after it."
           if live.get("prefix_warm_seconds") else ""),
        corpus_dependent=True,
        headline=("execution_accuracy", live["execution_accuracy"]),
        backend="ollama", model=NL2SQL_MODEL)


ollama reachable: True | cached SQL for 34/34 questions
prefix warmed in 0.9s (this call is discarded, not scored)
  correct                  How many Swiss decisions are from canton CH?
  correct                  How many Swiss decisions are from canton ZH?
  correct                  How many Swiss decisions are from canton GR?
  correct                  How many Swiss decisions are from canton BL?
  correct                  How many Swiss decisions are from canton AG?
  correct                  How many Swiss decisions are from canton BS?
  correct                  How many Swiss decisions are from canton SO?
  correct                  How many Swiss decisions are from canton BE?
  correct                  How many ECHR cases are against Russia?
  correct                  How many ECHR cases are against Poland?
  correct                  How many ECHR cases are against Norway?
  correct                  How many ECHR cases are against Germany?
  correct                  How many ECHR

## Tier 4 — Behavioural test
Does the system refuse what it cannot answer?


In [1]:
# TRAP-SET REFUSAL --------------------------------------------------------------------------
traps = [
    ("In what proportion of cases did the mother receive custody?", "refuse"),
    ("How many applicants were married?", "refuse"),
    ("Who received sole custody most often?", "refuse"),
    ("How many cases against Poland are in the corpus?", "answer"),
    ("How many merits judgments found a violation?", "answer"),
    ("How many Swiss decisions are from Kanton Bern?", "answer"),
]


def refuses(q):
    """The deployed refusal decision, in ask_anything()'s order -- same functions as the
    routing metric."""
    if _diachronic_trigger(q):
        return False
    if not aggregate_trigger(q):
        return False
    if _match_deployed_field(q):
        return False
    return bool(_UNEXTRACTED.search(q))


rows_t = [(q, exp, refuses(q)) for q, exp in traps]
correct_t = sum(1 for q, exp, r in rows_t if (r and exp == "refuse") or (not r and exp == "answer"))
# expected-refuse questions that were NOT refused. The previous version of this line ended in
# `and False`, which made the condition unsatisfiable and the metric a hardcoded zero.
false_fab = [q for q, exp, r in rows_t if exp == "refuse" and not r]
blocked_ok = [q for q, exp, r in rows_t if exp == "answer" and r]
for q, exp, r in rows_t:
    print(f"  expected {exp:6s} -> {'refused' if r else 'answered':8s} | {q}")
print(f"\nhandled correctly {correct_t}/{len(traps)} | not refused though unanswerable: "
      f"{len(false_fab)} | wrongly blocked: {len(blocked_ok)}")
for q in false_fab:
    print(f"  NOT REFUSED: {q}")

reg("trap_refusal", "Trap-set refusal",
    "Does the system refuse questions whose answer is NOT in the data (custody outcome, marital "
    "status) instead of fabricating a statistic -- while still letting the answerable ones "
    "through?",
    "A designed set of unanswerable vs answerable aggregate questions run through the deployed "
    "refusal chain (diachronic trigger -> aggregate trigger -> deployed field -> deny-list). No "
    "LLM: refusal is never delegated to the model.",
    {"n": len(traps), "handled_correctly": correct_t,
     "false_fabrications": len(false_fab),
     "false_fabrication_questions": false_fab or None,
     "wrongly_blocked": len(blocked_ok),
     "wrongly_blocked_questions": blocked_ok or None},
    "'...did the mother receive custody?' -> refused;  'cases against Poland' -> answered",
    "behavioural",
    f"{correct_t} of {len(traps)} traps are handled as designed. {len(false_fab)} unanswerable "
    f"question(s) are not caught by the deny-list"
    + (f" ({'; '.join(q[:60] for q in false_fab)})" if false_fab else "")
    + ". Those carry no lexical aggregate trigger either, so the dispatcher sends them to "
      "Bucket 1 rather than refusing them: the answer then depends on the grounding and "
      "citation-precision checks, not on the deny-list. A lost refusal, not a demonstrated "
      "fabrication -- and the measured limit of a lexical net. n=6 is a designed probe, not a "
      "sample: it demonstrates the mechanism and supports no rate.",
    corpus_dependent=False, headline=("handled_correctly", f"{correct_t}/{len(traps)}"))


  expected refuse -> refused  | In what proportion of cases did the mother receive custody?
  expected refuse -> refused  | How many applicants were married?
  expected refuse -> answered | Who received sole custody most often?
  expected answer -> answered | How many cases against Poland are in the corpus?
  expected answer -> answered | How many merits judgments found a violation?
  expected answer -> answered | How many Swiss decisions are from Kanton Bern?

handled correctly 5/6 | not refused though unanswerable: 1 | wrongly blocked: 0
  NOT REFUSED: Who received sole custody most often?
17. TRAP-SET REFUSAL
  QUESTION IT ANSWERS : Does the system refuse questions whose answer is NOT in the data (custody outcome, marital status) instead of fabricating a statistic -- while still letting the answerable ones through?
  HOW MEASURED        : A designed set of unanswerable vs answerable aggregate questions run through the deployed refusal chain (diachronic trigger -> aggregate trigger -

In [1]:
# ABSTENTION-TYPE DISTRIBUTION ----------------------------------------------------------------
# answer() carries five abstention layers (1 aggregate, 2 no-hits/off-topic, 3 boilerplate-only,
# 4 scope-empty, 4b source-empty). The trap-set metric tests the DISPATCHER's refusal; this asks
# which of the RETRIEVAL-side layers have ever actually fired, and on what.
#
# Two populations, and they answer different questions. The frozen retrieval set is 24 on-topic
# Bucket-1 queries, so it cannot exercise abstention -- a zero there is a property of the query
# set, not evidence that the layers work. The full provenance corpus is every grounded answer the
# pipeline has recorded, across configurations and dates: uncontrolled, but the only place the
# layers have fired.
ABSTAIN_LAYERS = {"type1_aggregate": "Layer 1 -- aggregate question (routed away upstream)",
                  "type2_off_topic": "Layer 2 -- nothing in the corpus is on topic",
                  "type3_boilerplate_only": "Layer 3 -- only communicated-case boilerplate matched",
                  "type4_scope_empty": "Layer 4 -- respondent-State scope left no passage",
                  "type4b_source_empty": "Layer 4b -- the source route left no passage"}
frozen = {"answers": len(PROV), "abstained": 0, "by_type": {}}
for d in PROV:
    if d["rec"].get("abstained"):
        frozen["abstained"] += 1
        t = d["rec"].get("abstain_type") or "unlabelled"
        frozen["by_type"][t] = frozen["by_type"].get(t, 0) + 1
allp = {"records": len(PROV_ALL), "abstained": 0, "by_type": {}}
_dates = sorted(x.get("asked_at", "") for x in PROV_ALL if x.get("asked_at"))
for rec in PROV_ALL:
    if rec.get("abstained"):
        allp["abstained"] += 1
        t = rec.get("abstain_type") or "unlabelled"
        allp["by_type"][t] = allp["by_type"].get(t, 0) + 1
allp["date_range"] = [_dates[0][:10], _dates[-1][:10]] if _dates else None
allp["abstention_rate"] = pct(allp["abstained"] / allp["records"]) if allp["records"] else None
allp["abstention_rate_ci95"] = ci(allp["abstained"], allp["records"]) if allp["records"] else None
never_fired = [k for k in ABSTAIN_LAYERS if k not in allp["by_type"]]

print(f"frozen retrieval set ({frozen['answers']} recorded answers over the 24 Bucket-1 "
      f"queries): {frozen['abstained']} abstained {frozen['by_type'] or ''}")
print(f"  -> zero by construction: every query in that set is an on-topic Bucket-1 question, so "
      f"no retrieval-side layer has anything to fire on")
print(f"full provenance corpus ({allp['records']} records, {allp['date_range']}): "
      f"{allp['abstained']} abstained = {allp['abstention_rate']} CI {allp['abstention_rate_ci95']}")
for t, k in sorted(allp["by_type"].items(), key=lambda x: -x[1]):
    print(f"    {k:3d}  {t}  -- {ABSTAIN_LAYERS.get(t, 'unlabelled layer')}")
print(f"  layers never observed firing: {never_fired}")
print("  type1_aggregate cannot appear here: ask_anything() routes aggregate questions to the "
      "query layer BEFORE answer() is reached, so that boundary is measured by the routing "
      "metric instead.")

reg("abstention_types", "Abstention-type distribution over the retrieval set",
    "When the retrieval path declines to answer, which layer stopped it -- and has each layer "
    "ever actually fired?",
    f"The abstain_type recorded on every grounded answer, over two populations: the "
    f"{frozen['answers']} answers of the frozen Bucket-1 query set (the same run the citation "
    f"metrics use), and all {allp['records']} provenance records the pipeline has ever written "
    f"({allp['date_range']}). The second is uncontrolled -- it spans configurations and dates -- "
    f"and is reported as such, because it is the only population in which the layers have fired. "
    f"Read from stored provenance; nothing is re-generated.",
    {"abstentions_recorded": allp["abstained"],
     "abstentions_recorded_of": allp["records"],
     "abstentions_recorded_ci95": allp["abstention_rate_ci95"],
     "frozen_retrieval_set": frozen,
     "frozen_set_note": "0 by construction -- all 24 queries are on-topic Bucket-1 questions, "
                        "so a zero here is a property of the query set, not evidence about the "
                        "layers",
     "full_provenance_corpus": allp,
     "layers_never_observed": never_fired,
     "layer_legend": ABSTAIN_LAYERS,
     "type1_note": "Layer 1 (aggregate) cannot appear in provenance: the dispatcher routes "
                   "aggregate questions to the query layer before answer() is reached. That "
                   "boundary is the routing metric.",
     "source_files": [str(CC), str(PROV_DIR)]},
    (f"{max(allp['by_type'], key=allp['by_type'].get)} fired "
     f"{max(allp['by_type'].values())} time(s)" if allp["by_type"] else "no abstention recorded"),
    "behavioural",
    f"On the frozen retrieval set the system abstained {frozen['abstained']}/{frozen['answers']} "
    f"times -- and that zero carries no information, because all 24 queries are on-topic "
    f"Bucket-1 questions by construction. Across all {allp['records']} recorded answers it "
    f"abstained {allp['abstained']} times ({allp['abstention_rate']}, "
    f"CI {allp['abstention_rate_ci95']}), split "
    f"{allp['by_type'] or 'not at all'}. "
    f"{len(never_fired)} of the {len(ABSTAIN_LAYERS)} layers have never been observed firing "
    f"({', '.join(never_fired) or 'none'}), so their behaviour is asserted by construction and "
    f"not by measurement -- an unexercised guard is not a demonstrated one. The wider corpus "
    f"mixes configurations and dates and is a deployment record, not a controlled sample.",
    corpus_dependent=True,
    headline=("abstentions_recorded", allp["abstained"]),
    backend=PROV_BACKEND, model=PROV_MODEL)


frozen retrieval set (48 recorded answers over the 24 Bucket-1 queries): 0 abstained 
  -> zero by construction: every query in that set is an on-topic Bucket-1 question, so no retrieval-side layer has anything to fire on
full provenance corpus (89 records, ['2026-08-19', '2026-09-11']): 5 abstained = 0.056 CI [0.024, 0.125]
      3  type2_off_topic  -- Layer 2 -- nothing in the corpus is on topic
      2  type3_boilerplate_only  -- Layer 3 -- only communicated-case boilerplate matched
  layers never observed firing: ['type1_aggregate', 'type4_scope_empty', 'type4b_source_empty']
  type1_aggregate cannot appear here: ask_anything() routes aggregate questions to the query layer BEFORE answer() is reached, so that boundary is measured by the routing metric instead.
18. ABSTENTION-TYPE DISTRIBUTION OVER THE RETRIEVAL SET
  QUESTION IT ANSWERS : When the retrieval path declines to answer, which layer stopped it -- and has each layer ever actually fired?
  HOW MEASURED        : The abstain_

## Summary


In [1]:
# SUMMARY ---------------------------------------------------------------------------------------
HANDOVER = [
    f"CONTENT FIELDS: {len(SCORABLE)} field(s) carry adjudicated labels and "
    f"{sorted(DEPLOYED_FIELDS) or 'none'} are deployed (parquet + sidecar, discoverable by the "
    f"dispatcher). Deploying the rest is field_deploy.ipynb's full-corpus pass: one LLM call per "
    f"candidate case with a per-case prompt, so prefix caching does not help -- ~60 s per case "
    f"on local CPU, i.e. hours to overnight per field; on the hosted backend the same work is "
    f"bounded by the per-minute token budget and lands in the low hours. That is the remaining "
    f"migration candidate.",
    "GOLD PROVENANCE: the field gold sets read here are model-adjudicated where their sidecar "
    "says so -- a second model reading the case evidence blind to the drafts, not a human "
    "annotator. Every field metric carries that provenance through. No human inter-annotator "
    "agreement exists for any field, so no claim of human validation can be made from these "
    "F1 figures.",
    "nl2sql() stays on local qwen2.5-coder:3b and must not be migrated: the hosted path is no "
    "faster for a constant KV-cached prefix (~9 min vs ~8 min for 30 questions), and the measured "
    "failure modes of a small local translator are the evidence for the deterministic nets. See "
    "the comment in the synthetic-QA cell.",
    "Bucket-1 generation stays on the hosted backend, where it already runs.",
    "NOT YET MEASURED: with the alienation branch retired and no deployed field, 17 of the 20 "
    "gold content-aggregate questions now reach the NL->SQL translator (see the dispatch confusion matrix). The share "
    "that nl2sql()'s own deny-list refuses before any model call is measured there; what the "
    "EXPLAIN / required-filter / GROUP BY nets do with the rest is not. Deploying a content "
    "field closes most of this by routing those questions to Bucket 3 instead.",
    "retrieval_judgments.csv was judged on the pre-September index. Metrics 4 and 5 report the "
    "measured overlap with the current ranking and mark themselves STALE below 0.90. Refreshing "
    "means regenerating the template from retrieval_evaluation.ipynb and re-judging -- old "
    "labels cannot be reused, because the retrieved chunk ids differ and do not join.",
]

print("=" * 80)
print("SUMMARY -- every claim tested with no recruited evaluators")
print("-" * 80)
for k in ORDER:
    r = RESULTS[k]
    flag = "" if r["status"] == "computed" else f"  <{r['status']}>"
    print(f"  {r['name']:64s} [{r['tier']}]{flag}")
    print(f"      headline {r['headline_key']} = {r['headline']} | "
          f"corpus_dependent={r['corpus_dependent']}"
          + (f" | {r['backend']}/{r['model']}" if r["backend"] else ""))

out = {"generated": RUN_AT, "run_environment": RUN_ENV, "handover": HANDOVER,
       "metrics": [dict(key=k, order=i + 1, **RESULTS[k]) for i, k in enumerate(ORDER)]}
(REPORTS / "evaluation_results.json").write_text(json.dumps(out, indent=1, default=str))
print(f"\nsaved -> {REPORTS / 'evaluation_results.json'}")


_NUM_PREFIX = re.compile(r"^\d+\.\s*")


def _short(name):
    """The name without its registration number -- the table numbers its own rows."""
    return _NUM_PREFIX.sub("", name)


def _fmt_ci(v):
    """The 95% Wilson interval belonging to a metric's headline, when it has one.

    By convention every rate in a result dict stores its interval under <key>_ci95, so this is a
    lookup rather than a guess. Metrics whose headline is not a rate (ECE, P@k, a threshold) have
    no interval and say so with a dash instead of borrowing someone else's."""
    c = v["value"].get(f"{v['headline_key']}_ci95")
    return f"[{c[0]}, {c[1]}]" if c else "--"


lines = [f"# Evaluation summary\n\n",
         f"Generated {RUN_AT} by `src/evaluation.ipynb`. Every number here is recomputed by that "
         f"notebook; none is a literal.\n\n",
         f"- index: **{RUN_ENV['index_passages']}** passages | ECHR query DB: "
         f"**{RUN_ENV['echr_cases_in_db']}** cases\n",
         f"- generation: `{RUN_ENV['generation']['backend']}` / "
         f"`{RUN_ENV['generation']['model']}` (ready={RUN_ENV['generation']['ready']})\n",
         f"- NL->SQL: `ollama` / `{RUN_ENV['nl2sql']['model']}` "
         f"(reachable at boot={RUN_ENV['nl2sql']['reachable_at_boot']})\n",
         f"- deployed content fields: "
         f"{RUN_ENV['deployed_content_fields'] or 'none yet'}\n\n",
         "`corpus_dependent = false` means a frozen question set or per-case labels the "
         "September 2026 re-import cannot touch; `true` means the number is computed against the "
         "corpus as it currently stands, and `STALE` means it rests on judgments made against a "
         "superseded index.\n\n",
         "| # | metric | tier | headline | 95% CI | corpus-dependent | status |\n",
         "|---|---|---|---|---|---|---|\n"]
for i, k in enumerate(ORDER, 1):
    v = RESULTS[k]
    head = ("pending" if v["status"] == "pending"
            else f"`{v['headline_key']}` = **{v['headline']}**")
    lines.append(f"| {i} | {_short(v['name'])} | {v['tier']} | {head} | {_fmt_ci(v)} | "
                 f"{str(v['corpus_dependent']).lower()} | {v['status']} |\n")

lines.append("\n## Takeaways\n\n")
for i, k in enumerate(ORDER, 1):
    v = RESULTS[k]
    lines.append(f"**{v['name']}** — {v['takeaway']}\n\n")

_pending = [RESULTS[k] for k in ORDER if RESULTS[k]["status"] == "pending"]
_stale = [RESULTS[k] for k in ORDER if RESULTS[k]["status"] == "STALE"]
lines.append("## What did not run, and why\n\n")
if not _pending and not _stale:
    lines.append("Every metric ran and none is stale.\n\n")
for v in _pending:
    lines.append(f"- **PENDING — {v['name']}**: {v['value']['pending_reason']}"
                 + (f" *Unblocked by:* {v['value']['unblocked_by']}"
                    if v["value"].get("unblocked_by") else "") + "\n")
for v in _stale:
    lines.append(f"- **STALE — {v['name']}**: judged on the pre-re-import index. Only "
                 f"{v['value'].get('current_top6_overlap_with_judged_pool')} of what the current "
                 f"index returns for these queries was ever judged, so the number stands for "
                 f"that index only. Refresh: regenerate the template from "
                 f"`retrieval_evaluation.ipynb` and re-judge -- old labels do not join.\n")
lines.append("\n## Handover\n\n")
for h in HANDOVER:
    lines.append(f"- {h}\n")
(REPORTS / "evaluation_summary.md").write_text("".join(lines), encoding="utf-8")
print(f"saved -> {REPORTS / 'evaluation_summary.md'}")
print(f"  {len(ORDER)} metrics | {len(_pending)} pending | {len(_stale)} stale")


SUMMARY -- every claim tested with no recruited evaluators
--------------------------------------------------------------------------------
  1. Routing accuracy and the error directions                     [gold]
      headline route_accuracy = 0.812 | corpus_dependent=False
  2. Dispatch confusion matrix (4x4)                               [gold]
      headline strict_4-way_accuracy = 0.675 | corpus_dependent=False
  3. Confidence calibration                                        [gold]
      headline ece_calibrated_out_of_fold@10bins = 0.084 | corpus_dependent=False
  4. Confidence granularity behind the ECE                         [gold]
      headline distinct_raw_confidence_values = 14 | corpus_dependent=False
  5. Retrieval quality                                             [gold]  <STALE>
      headline P@6 = 0.451 | corpus_dependent=True
  6. Retrieval precision by language, jurisdiction and ECHR section [gold]  <STALE>
      headline law_share_of_echr_hits = 0.229 | corpus_